# ApexID: closed-set retrieval and representation benchmark

This updated Stage 02 consumes frozen Stage 00 splits and Stage 01 embedding caches.
It runs on CPU and requires **no model training, image inference, embedding extraction,
split generation, rejection calibrator, or deployment threshold selection**.

The original image-gallery cosine retrieval function is preserved byte for byte:
Rank-1/5/10, CMC@20/50, mAP, and identity-balanced Rank-1 retain their definitions.
Model names are read from the completed Stage 01 manifest, supporting the three-model
bundle in the original notebook and the four-model bundle in the latest results.

New metrics use the **same closed-set gallery and known-query identities**. They do
not evaluate unknown rejection or novel-identity discovery. Saved historical outputs
have been cleared because they do not belong to this updated implementation.

Run all cells in order. Attach the same Stage 00/01 bundles. Dependencies are numpy,
pandas, scipy, scikit-learn and matplotlib (normally present on Kaggle).
The configuration cell controls sampling limits, explicit species metadata, and plots.


In [16]:
# ============================================================
# 1. Minimal local CPU environment and explicit paths
# ============================================================

import ast
import json
import math
import os
import platform
import shutil
import sys
import tempfile
import time
from datetime import datetime, timezone
from pathlib import Path

ON_KAGGLE = Path("/kaggle/working").is_dir()

def _is_usable_path(path: Path | None) -> bool:
    if path is None:
        return False
    text = str(path).strip()
    if not text:
        return False
    # Never treat a Windows drive path as valid on Linux/Kaggle.
    if ON_KAGGLE and (":\\" in text or text.startswith(("D:\\", "C:\\", "d:\\", "c:\\"))):
        return False
    return True


def _first_existing_file(candidates: list[Path]) -> Path | None:
    for path in candidates:
        if _is_usable_path(path) and path.is_file():
            return path.resolve()
    return None


def _first_existing_root(candidates: list[Path], marker: str) -> Path | None:
    for path in candidates:
        if not _is_usable_path(path):
            continue
        if (path / marker).is_file():
            return path.resolve()
    return None


def _search_kaggle_input(marker: str, preferred_dirname: str | None = None) -> Path | None:
    """Find a Stage root under /kaggle/input regardless of owner/slug nesting."""
    root = Path("/kaggle/input")
    if not root.is_dir():
        return None
    preferred_hits: list[Path] = []
    other_hits: list[Path] = []
    for candidate in root.rglob(marker):
        if not candidate.is_file():
            continue
        parent = candidate.parent.resolve()
        if preferred_dirname and parent.name == preferred_dirname:
            preferred_hits.append(parent)
        else:
            other_hits.append(parent)
    hits = preferred_hits or other_hits
    return hits[0] if hits else None


STAGE02_NOTEBOOK_PATH = _first_existing_file([
    Path(os.environ.get("APEXID_STAGE02_NOTEBOOK", "")),
    Path("/kaggle/working/02_run_cvpr_benchmark.ipynb"),
    Path.cwd() / "02_run_cvpr_benchmark_closed_set.ipynb",
    Path.cwd() / "02_run_cvpr_benchmark.ipynb",
    *([] if ON_KAGGLE else [
        Path(r"D:\CVPR_BM\new_bm\02_run_cvpr_benchmark.ipynb"),
        Path(r"c:\Users\Legion 5\Desktop\ApexID\notebooks\02_run_cvpr_benchmark.ipynb"),
    ]),
])
if STAGE02_NOTEBOOK_PATH is None:
    STAGE02_NOTEBOOK_PATH = Path("02_run_cvpr_benchmark.ipynb")

STAGE00_CANDIDATES = [
    Path(os.environ.get("APEXID_PREPARED_INPUT") or os.environ.get("APEXID_PREPARED_ROOT") or ""),
    Path("/kaggle/input/datasets/seifelkerdany/apexid-cvpr-prepared-bundle/apexid_cvpr_prepared"),
    Path("/kaggle/input/apexid-cvpr-prepared-bundle/apexid_cvpr_prepared"),
    Path("/kaggle/input/datasets/seifelkerdany/apexid-cvpr-prepared-bundle"),
    Path("/kaggle/working/apexid_cvpr_prepared"),
]
if not ON_KAGGLE:
    STAGE00_CANDIDATES.append(Path(r"D:\CVPR_BM\apexid_cvpr_prepared"))
STAGE00_ROOT = (
    _first_existing_root(STAGE00_CANDIDATES, "preparation_manifest.json")
    or _search_kaggle_input("preparation_manifest.json", preferred_dirname="apexid_cvpr_prepared")
)
if STAGE00_ROOT is None:
    raise FileNotFoundError(
        "Missing frozen Stage 00 root (preparation_manifest.json). "
        "On Kaggle: Add Input â†’ attach the Stage 00 prepared bundle "
        "(apexid-cvpr-prepared-bundle / apexid_cvpr_prepared). "
        "Or set APEXID_PREPARED_INPUT to that folder."
    )

STAGE01_CANDIDATES = [
    Path(os.environ.get("APEXID_STAGE01_INPUT", "")),
    Path("/kaggle/working/apexid_cvpr_stage01_bhca_logistic"),
    Path("/kaggle/input/apexid-cvpr-stage01-bhca-logistic"),
    Path("/kaggle/input/apexid-cvpr-stage01-bhca-logistic/apexid_cvpr_stage01_bhca_logistic"),
    Path("/kaggle/input/datasets/mohamedbassat/apexid-cvpr-stage01-bhca-logistic"),
    Path("/kaggle/input/datasets/mohamedbassat/apexid-cvpr-stage01-bhca-logistic/apexid_cvpr_stage01_bhca_logistic"),
]
if not ON_KAGGLE:
    STAGE01_CANDIDATES.append(Path(r"D:\CVPR_BM\apexid_cvpr_stage01_bhca_logistic"))
STAGE01_ROOT = (
    _first_existing_root(STAGE01_CANDIDATES, "stage01_manifest.json")
    or _search_kaggle_input("stage01_manifest.json", preferred_dirname="apexid_cvpr_stage01_bhca_logistic")
)
if STAGE01_ROOT is None:
    raise FileNotFoundError(
        "Missing frozen Stage 01 root (stage01_manifest.json). "
        "On Kaggle: Add Input â†’ apexid-cvpr-stage01-bhca-logistic. "
        "Or set APEXID_STAGE01_INPUT to the folder that contains stage01_manifest.json."
    )

OUTPUT_ROOT = Path(os.environ.get(
    "APEXID_STAGE02_OUTPUT",
    "/kaggle/working/apexid_cvpr_closed_representation" if ON_KAGGLE
    else str(Path.cwd() / "apexid_cvpr_closed_representation"),
)).resolve()
WORKSPACE_ROOT = Path.cwd()
# No outer process pool is used. BLAS owns this single configured level of
# parallelism, avoiding nested joblib/OpenMP oversubscription.
N_JOBS = int(os.environ.get("APEXID_N_JOBS", "-1"))
CPU_COUNT = os.cpu_count() or 1
NUMERIC_THREADS = CPU_COUNT if N_JOBS == -1 else max(1, min(N_JOBS, CPU_COUNT))
for _variable in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ.setdefault(_variable, str(NUMERIC_THREADS))

import numpy as np
import pandas as pd
try:
    from IPython.display import Markdown, display
except ImportError:
    Markdown = str
    def display(value):
        print(value)


BENCHMARK_WALL_START = time.perf_counter()

print("Stage 02 notebook:", STAGE02_NOTEBOOK_PATH)
print("Stage 00:", STAGE00_ROOT)
print("Stage 01:", STAGE01_ROOT)
print("Final output:", OUTPUT_ROOT)
print("CPU count:", CPU_COUNT, "configured N_JOBS:", N_JOBS)
print("Stage 02 requires no model inference and no GPU.")


Stage 02 notebook: 02_run_cvpr_benchmark.ipynb
Stage 00: /kaggle/input/datasets/seifelkerdany/apexid-cvpr-prepared-bundle/apexid_cvpr_prepared
Stage 01: /kaggle/input/datasets/mohamedbassat/apexid-cvpr-stage01-bhca-logistic/apexid_cvpr_stage01_bhca_logistic
Final output: /kaggle/working/apexid_cvpr_closed_representation
CPU count: 4 configured N_JOBS: -1
Stage 02 requires no model inference and no GPU.


## Configuration
Use the same frozen input bundles. Adjust verified species labels only when metadata require it.

In [17]:
# Models to benchmark: keys must match Stage 01 folder names, e.g.
# embeddings/ApexID_Convnext_Reweighted/jaguar_antonio/embeddings.npz
# Override without editing: APEXID_STAGE02_MODELS=ApexID_Convnext_Reweighted
# Set MODELS_TO_EVALUATE = None to run every entry in stage01_manifest.json models_completed.
_STAGE02_MODELS_ENV = os.environ.get("APEXID_STAGE02_MODELS", "").strip()
if _STAGE02_MODELS_ENV:
    MODELS_TO_EVALUATE = tuple(m.strip() for m in _STAGE02_MODELS_ENV.split(",") if m.strip())
else:
    MODELS_TO_EVALUATE = ("ApexID_Convnext_Reweighted",)

EXCLUDED_LEGACY_MODELS = frozenset({"ApexID_v1", "ApexID_HA_v2", "ApexID_v2", "MiewIDv3", "MegaDescriptor"})
EXTERNAL_DATASETS = ("bristol_gorillas", "elephants", "gcn_id", "jaguar_antonio")
SPLIT_SEEDS = (42, 123, 3407, 2026, 1337)
STAGE02_VERSION = "apexid-closed-representation-1.0.0"
CLOSED_SET_PRE_LOGISTIC_FUNCTION_SHA256 = '59d5877444690a393eb70f385e9981520c8e4cc95d0678e4ca2196880813eea1'
REP_IMPLEMENTATION_SHA256 = '00019506ec971cf74756239555bbd0b1fdb9e1aff6640f06d82bdca40121b91b'
BOOTSTRAP_REPLICATES = int(os.environ.get("APEXID_BOOTSTRAP_REPLICATES", "1000"))
BOOTSTRAP_SEED = 20260911
SIMILARITY_CHUNK_SIZE = int(os.environ.get("APEXID_SIMILARITY_CHUNK", "4096"))
RUN_PAIRED_COMPARISONS = os.environ.get("APEXID_RUN_PAIRED", "1") == "1"
RUN_PLOTS = os.environ.get("APEXID_RUN_PLOTS", "1") == "1"
REP_SEED = 20261006

# These must encode species, not dataset identifiers or arbitrary categories.
SPECIES_COLUMNS = ("species", "species_name", "cache_species")
OBSERVATION_GROUP_COLUMNS = ("observation_group", "observation_group_id")
# Use only verified single-species datasets here when species columns are absent.
# Example syntax: {"your_dataset_name": "its_verified_species_label"}
VERIFIED_DATASET_SPECIES = {}
RUN_POOLED_SPECIES_DIAGNOSTICS = True
EXPORT_PAIR_SCORES = False  # True also saves sampled_pairs.npz per split/model.
REP_CONFIG = {
    "queries_per_identity": 16,
    "max_queries": 2048,
    "positive_pairs_per_query": 32,
    "negative_pairs_per_query": 64,
    "similarity_chunk": 128,
    "far_targets": [0.01],
    "geometry_per_identity": 12,
    "geometry_max_samples": 1024,
    "uniformity_t": 2.0,
    "known_k_clustering": True,
}
if BOOTSTRAP_REPLICATES < 2 or SIMILARITY_CHUNK_SIZE < 1:
    raise ValueError("Bootstrap replicates must be >=2 and chunk size must be positive")
for name in ("queries_per_identity", "max_queries", "positive_pairs_per_query", "negative_pairs_per_query", "similarity_chunk", "geometry_per_identity", "geometry_max_samples"):
    if REP_CONFIG[name] < 1:
        raise ValueError(f"{name} must be positive")
if not REP_CONFIG["far_targets"] or any(not 0 < f < 1 for f in REP_CONFIG["far_targets"]):
    raise ValueError("FAR targets must lie strictly between zero and one")
if REP_CONFIG["uniformity_t"] <= 0:
    raise ValueError("uniformity_t must be positive")



## Diagnostic definitions and interpretation

All embeddings are L2-normalized before the new metrics. Cosine similarity is
\(s(i,j)=z_i^Tz_j\). Results remain on their native scales (probabilities are 0â€“1,
cosine gaps are not percentages). `representation_metric_registry.csv` records units.

| Output | Definition / interpretation |
|---|---|
| Verification AUROC, `all` | Same-identity queryâ€“gallery pairs versus different-identity pairs. |
| Verification AUROC, `same_species` | Same-identity pairs versus different identities with the same explicit species label. A separate per-species table and species-macro AUROC are also saved. |
| TAR@FAR=0.01 | Maximum observed TAR at weighted empirical FAR â‰¤0.01, with no interpolation or splitting tied scores. Achieved FAR is reported. This is a **descriptive test ROC point**, not validation calibration or a deployment threshold. |
| Similarity distributions | Positive, all-negative, same-species-negative and cross-species-negative cosine means, quantiles and weighted histograms. |
| Identity gap | Mean positive similarity minus mean different-identity similarity; also reported with same-species negatives. Both groups use the same eligible query anchors. |
| Species gap | Mean same-species-negative similarity minus mean cross-species-negative similarity, on matched eligible anchors. A larger value alone does not prove better identity recognition. |
| Hard-negative margin | Per query: exact mean cosine over eligible gallery positives minus maximum cosine over eligible negatives. Same-species and unrestricted forms; identity-balanced mean, median, 10th percentile, and fraction â‰¤0. |
| Alignment | Identity-balanced mean squared L2 distance of sampled positive pairs: exactly `2 - 2*positive_cosine_mean`. Redundant with that mean, so not independent evidence. |
| Identity silhouette | Cosine silhouette using true identity labels; identity-balanced. A second version is computed **within each species**, then averaged equally by identity and species. Singleton identities are excluded from the final silhouette average; coverage is reported. |
| Species silhouette | Cosine silhouette by explicit species labels, averaged by species; pooled diagnostic only where multiple species are available. |
| Effective rank | `exp(-sum(p*log(p)))`, where p is the normalized singular-value spectrum. Both uncentered (RankMe-style) and centered variants, plus fractions of the sample/dimension ceiling. Rank alone does not imply identity information. |
| Uniformity | `log(mean(exp(-t * squared_L2_distance)))` over distinct unordered pairs in the geometry sample; t=2. Global uniformity is a diagnostic, not a universal goal for hierarchical embeddings. |
| Closed-set BÂ³ / ARI | Average-linkage clustering with cosine distances on the geometry sample, cut to the **known count K of represented identities**. BÂ³ uses item-averaged precision/recall; ARI is chance-adjusted. This uses known class-count information and is a conditional geometry diagnostic. |

### Sampling, fairness and missing metadata

- The exact original retrieval metrics still use all frozen gallery/query images.
- New pair diagnostics cap anchors per identity and positives/negatives per anchor.
  Negative sampling cycles through different identities before adding more images.
  Pair means/ROC use equal identity, equal query, then equal pair weights. A positive
  and negative ROC comparison requires both pair classes for the same anchors.
- Hard-negative maxima and positive means use the entire eligible gallery for each
  sampled query, not just the sampled diagnostic pairs.
- Self-pairs are excluded. Equal nonempty observation groups are excluded from pair
  metrics. If group metadata are absent, only image disjointness can be checked.
- Geometry/clustering use a bounded, identity-balanced sample from the closed gallery
  and query union. This sample can contain multiple observations from the same group;
  it is not an independent-observation estimate. Singleton and sample counts are saved.
- Sampling seeds exclude model names. Image-membership fingerprints must match across
  models. Caches bind scores to embedding bytes, labels, sampling configuration and code.
- Species labels are taken from explicit canonical/split columns, a cache `species`
  array, or `VERIFIED_DATASET_SPECIES`. Missing labels are **not guessed** from filenames,
  dataset names or training categories. Generic within-dataset metrics still run;
  unsupported species metrics are NaN with coverage/status, not zero.
- The optional pooled species diagnostic combines the already selected closed-set
  samples and namespaces identity/image IDs by dataset. It is excluded from headline
  dataset-macro averages. Species and dataset acquisition cues may be confounded.
- New summaries report split means/SD and all-pair model differences. They do **not**
  claim confidence intervals from the five overlapping evaluation splits. The original
  retrieval bootstrap is retained and remains conditional on frozen checkpoints.
- A headline diagnostic macro is available only when every configured dataset has
  complete split coverage. Partial available-dataset averages are separately labeled.

The manifest does not establish whether test identities were absent from backbone
training; that generalization claim requires training-data provenance.

References: [FaceNet](https://arxiv.org/abs/1503.03832),
[alignment/uniformity](https://proceedings.mlr.press/v119/wang20k.html),
[RankMe](https://proceedings.mlr.press/v202/garrido23a.html),
[silhouette](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_samples.html).


In [18]:
from __future__ import annotations
import contextlib, dataclasses, hashlib, json, math, os, platform, random, sys, tempfile, time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Callable, Mapping, Sequence
import numpy as np
import pandas as pd
def _json_default(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if dataclasses.is_dataclass(value):
        return dataclasses.asdict(value)
    if isinstance(value, set):
        return sorted(value)
    raise TypeError(f"Cannot serialize {type(value).__name__}")

def stable_json(value: Any) -> str:
    return json.dumps(
        value,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False,
        default=_json_default,
    )

def sha256_bytes(value: bytes) -> str:
    return hashlib.sha256(value).hexdigest()

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            block = handle.read(chunk_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()

def atomic_write_text(path: Path, text: str) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, temporary_name = tempfile.mkstemp(prefix=f".{path.name}.", dir=path.parent)
    os.close(fd)
    temporary = Path(temporary_name)
    try:
        temporary.write_text(text, encoding="utf-8")
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)

def write_json(path: Path, value: Any) -> None:
    atomic_write_text(Path(path), json.dumps(value, indent=2, sort_keys=True, default=_json_default) + "\n")

def atomic_write_csv(path: Path, frame: pd.DataFrame) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, temporary_name = tempfile.mkstemp(prefix=f".{path.name}.", dir=path.parent)
    os.close(fd)
    temporary = Path(temporary_name)
    try:
        frame.to_csv(temporary, index=False, lineterminator="\n")
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)

def atomic_save_npz(path: Path, **arrays: Any) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, temporary_name = tempfile.mkstemp(prefix=f".{path.stem}.", suffix=".npz", dir=path.parent)
    os.close(fd)
    temporary = Path(temporary_name)
    try:
        np.savez_compressed(temporary, **arrays)
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)

@dataclass
class RuntimeRecorder:
    rows: list[dict[str, Any]] = field(default_factory=list)

    @contextlib.contextmanager
    def measure(self, stage: str, **metadata: Any):
        start = time.perf_counter()
        try:
            yield
        finally:
            row = {"stage": stage, "seconds": time.perf_counter() - start, **metadata}
            self.rows.append(row)

    def record(self, stage: str, seconds: float, **metadata: Any) -> None:
        self.rows.append({"stage": stage, "seconds": float(seconds), **metadata})

    def frame(self) -> pd.DataFrame:
        base = [
            "stage",
            "model",
            "dataset",
            "split_seed",
            "seconds",
            "cache_hit",
            "number_images",
            "number_queries",
            "number_gallery_identities",
        ]
        frame = pd.DataFrame(self.rows)
        for column in base:
            if column not in frame:
                frame[column] = np.nan
        return frame[base + [c for c in frame.columns if c not in base]]

    def save(self, path: Path) -> None:
        atomic_write_csv(path, self.frame())

def stable_seed(*parts: Any) -> int:
    digest = hashlib.sha256("::".join(map(str, parts)).encode("utf-8")).digest()
    return int.from_bytes(digest[:8], byteorder="big", signed=False) % (2**32 - 1)

def _package_version(name: str) -> str | None:
    try:
        from importlib.metadata import version

        return version(name)
    except Exception:
        return None

def environment_provenance() -> dict[str, Any]:
    """Record CPU evaluation dependencies without importing accelerator/model libraries."""
    return {
        "python": sys.version,
        "platform": platform.platform(),
        "os": os.name,
        "execution_device": "CPU",
        "accelerator_runtime_queried": False,
        "packages": {
            name: _package_version(name)
            for name in ("numpy", "pandas", "scipy", "scikit-learn")
        },
    }

def l2_normalize(embeddings: np.ndarray) -> np.ndarray:
    values = np.asarray(embeddings, dtype=np.float32)
    if values.ndim != 2 or not np.isfinite(values).all():
        raise ValueError("Embeddings must be a finite two-dimensional array")
    norms = np.linalg.norm(values, axis=1, keepdims=True)
    if np.any(norms <= 1e-12):
        raise ValueError("Embedding cache contains zero-norm rows")
    return values / norms

def load_embedding_cache(
    path: Path,
    *,
    expected_metadata: Mapping[str, Any] | None = None,
    expected_image_ids: Sequence[str] | None = None,
) -> tuple[np.ndarray, dict[str, Any], np.ndarray, np.ndarray, np.ndarray]:
    with np.load(Path(path), allow_pickle=False) as cached:
        metadata = json.loads(str(cached["metadata_json"].item()))
        embeddings = l2_normalize(cached["embeddings"])
        image_ids = cached["image_ids"].astype(str)
        paths = cached["paths"].astype(str)
        identities = cached["identities"].astype(str)
    if expected_metadata is not None and stable_json(metadata) != stable_json(dict(expected_metadata)):
        raise ValueError("Embedding cache provenance mismatch")
    if expected_image_ids is not None and not np.array_equal(image_ids, np.asarray(expected_image_ids).astype(str)):
        raise ValueError("Embedding cache image order mismatch")
    if not (len(embeddings) == len(image_ids) == len(paths) == len(identities)):
        raise ValueError("Embedding cache array lengths differ")
    expected_dimension = int(metadata["embedding_dimension"])
    if embeddings.shape[1] != expected_dimension:
        raise ValueError(
            f"Embedding cache dimension {embeddings.shape[1]} != provenance {expected_dimension}"
        )
    return embeddings, metadata, image_ids, paths, identities

def save_closed_result_cache(
    path: Path,
    *,
    metadata: Mapping[str, Any],
    metrics: Mapping[str, Any],
    query_details: pd.DataFrame,
) -> None:
    required = {"image_id", "identity", "valid", "rank1", "average_precision"}
    if not required <= set(query_details):
        raise ValueError(f"Closed result details missing {sorted(required - set(query_details))}")
    atomic_save_npz(
        path,
        metadata_json=np.asarray(stable_json(dict(metadata))),
        metrics_json=np.asarray(stable_json(dict(metrics))),
        image_ids=query_details["image_id"].astype(str).to_numpy(dtype="U"),
        identities=query_details["identity"].astype(str).to_numpy(dtype="U"),
        valid=query_details["valid"].astype(bool).to_numpy(),
        rank1=query_details["rank1"].astype(float).to_numpy(),
        average_precision=query_details["average_precision"].astype(float).to_numpy(),
    )

def load_closed_result_cache(
    path: Path,
    *,
    expected_metadata: Mapping[str, Any] | None = None,
) -> tuple[dict[str, Any], pd.DataFrame]:
    with np.load(path, allow_pickle=False) as cached:
        metadata = json.loads(str(cached["metadata_json"].item()))
        metrics = json.loads(str(cached["metrics_json"].item()))
        details = pd.DataFrame(
            {
                "image_id": cached["image_ids"].astype(str),
                "identity": cached["identities"].astype(str),
                "valid": cached["valid"].astype(bool),
                "rank1": cached["rank1"].astype(float),
                "average_precision": cached["average_precision"].astype(float),
            }
        )
    if expected_metadata is not None and stable_json(metadata) != stable_json(dict(expected_metadata)):
        raise ValueError("Closed result cache provenance mismatch")
    return metrics, details

def closed_set_retrieval_metrics(
    *,
    query_embeddings: np.ndarray,
    query_identities: Sequence[str],
    gallery_embeddings: np.ndarray,
    gallery_identities: Sequence[str],
    chunk_size: int = 1024,
) -> tuple[dict[str, float], pd.DataFrame]:
    """Repository-equivalent retrieval metrics without importing PyTorch."""
    query = np.asarray(query_embeddings).astype(np.float64, copy=False)
    gallery = np.asarray(gallery_embeddings).astype(np.float64, copy=False)
    if query.ndim != 2 or gallery.ndim != 2:
        raise ValueError("embeddings must have shape (num_samples, embedding_dim).")
    if query.shape[1:] != gallery.shape[1:]:
        raise ValueError("query and gallery embeddings must have the same embedding dimension.")
    query = (query / np.maximum(np.linalg.norm(query, axis=1, keepdims=True), 1e-12)).astype(np.float32, copy=False)
    gallery = (gallery / np.maximum(np.linalg.norm(gallery, axis=1, keepdims=True), 1e-12)).astype(np.float32, copy=False)
    query_labels = np.asarray(query_identities).astype(str)
    gallery_labels = np.asarray(gallery_identities).astype(str)
    if query_labels.ndim != 1 or len(query_labels) != len(query):
        raise ValueError("query identity labels do not align with query embeddings")
    if gallery_labels.ndim != 1 or len(gallery_labels) != len(gallery):
        raise ValueError("gallery identity labels do not align with gallery embeddings")
    if int(chunk_size) <= 0:
        raise ValueError("chunk_size must be > 0.")

    top_k = (1, 5, 10)
    cmc_ranks = (20, 50)
    ranks = tuple(sorted(set(top_k) | set(cmc_ranks)))
    result = {"known/mAP": float("nan"), "known/BAKS": float("nan")}
    result.update({f"known/top{k}": float("nan") for k in top_k})
    result.update({f"known/cmc_rank{k}": float("nan") for k in cmc_ranks})
    num_queries = int(len(query_labels))
    num_gallery = int(len(gallery_labels))
    query_values = {
        "valid": np.zeros(num_queries, dtype=bool),
        "rank1": np.full(num_queries, np.nan, dtype=np.float64),
        "average_precision": np.full(num_queries, np.nan, dtype=np.float64),
    }
    if num_queries and num_gallery:
        max_rank = max(ranks)
        top_hits = {k: 0 for k in ranks}
        valid_queries = 0
        ap_values = []
        top1_correct = np.zeros(num_queries, dtype=np.float64)
        positive_indices = {
            label: np.flatnonzero(gallery_labels == label) for label in np.unique(query_labels)
        }
        positive_counts = {label: int(indices.size) for label, indices in positive_indices.items()}
        for start in range(0, num_queries, int(chunk_size)):
            stop = min(start + int(chunk_size), num_queries)
            labels_chunk = query_labels[start:stop]
            similarity = query[start:stop] @ gallery.T
            has_positive = np.asarray([positive_counts.get(label, 0) > 0 for label in labels_chunk], dtype=bool)
            if not np.any(has_positive):
                continue
            valid_queries += int(np.sum(has_positive))
            query_values["valid"][start:stop] = has_positive
            rank_limit = min(max_rank, num_gallery)
            if rank_limit == num_gallery:
                top_indices = np.argsort(-similarity, axis=1)[:, :rank_limit]
            else:
                top_indices = np.argpartition(-similarity, kth=rank_limit - 1, axis=1)[:, :rank_limit]
                top_scores = np.take_along_axis(similarity, top_indices, axis=1)
                top_order = np.argsort(-top_scores, axis=1)
                top_indices = np.take_along_axis(top_indices, top_order, axis=1)
            top_relevance = gallery_labels[top_indices] == labels_chunk[:, None]
            top1_correct[start:stop] = top_relevance[:, :1].any(axis=1).astype(np.float64)
            query_values["rank1"][start:stop][has_positive] = top1_correct[start:stop][has_positive]
            for k in ranks:
                top_hits[k] += int(top_relevance[has_positive, :min(k, rank_limit)].any(axis=1).sum())

            valid_local = np.flatnonzero(has_positive)
            sorted_scores = -np.sort(-similarity[valid_local], axis=1)
            for sorted_row, row_offset in enumerate(valid_local):
                matches = positive_indices[labels_chunk[row_offset]]
                scores = similarity[row_offset, matches]
                score_order = np.argsort(-scores)
                scores = scores[score_order]
                positive_ranks = np.searchsorted(-sorted_scores[sorted_row], -scores, side="left")
                if positive_ranks.size > 1:
                    tie_offsets = np.zeros(positive_ranks.shape[0], dtype=np.int64)
                    tie_start = 0
                    while tie_start < scores.size:
                        tie_stop = tie_start + 1
                        while tie_stop < scores.size and scores[tie_stop] == scores[tie_start]:
                            tie_stop += 1
                        tie_offsets[tie_start:tie_stop] = np.arange(tie_stop - tie_start, dtype=np.int64)
                        tie_start = tie_stop
                    positive_ranks = positive_ranks + tie_offsets
                positive_ranks = positive_ranks + 1
                positive_ranks.sort()
                ap = float(np.mean(np.arange(1, positive_ranks.size + 1, dtype=np.float64) / positive_ranks))
                ap_values.append(ap)
                query_values["average_precision"][start + row_offset] = ap

        if valid_queries:
            for k in top_k:
                result[f"known/top{k}"] = float(top_hits[k] / valid_queries)
            for k in cmc_ranks:
                result[f"known/cmc_rank{k}"] = float(top_hits[k] / valid_queries)
            result["known/mAP"] = float(np.mean(ap_values)) if ap_values else float("nan")
            per_identity = [float(np.mean(top1_correct[query_labels == label])) for label in np.unique(query_labels)]
            result["known/BAKS"] = float(np.mean(per_identity)) if per_identity else float("nan")

    point = {
        "Rank-1": float(result["known/top1"]),
        "Rank-5": float(result["known/top5"]),
        "Rank-10": float(result["known/top10"]),
        "CMC@20": float(result["known/cmc_rank20"]),
        "CMC@50": float(result["known/cmc_rank50"]),
        "mAP": float(result["known/mAP"]),
        "identity-balanced Rank-1": float(result["known/BAKS"]),
        "effective_rank_5": int(min(5, num_gallery)),
        "effective_rank_10": int(min(10, num_gallery)),
        "effective_rank_20": int(min(20, num_gallery)),
        "effective_rank_50": int(min(50, num_gallery)),
    }
    details = pd.DataFrame({
        "identity": query_labels,
        "valid": query_values["valid"],
        "rank1": query_values["rank1"],
        "average_precision": query_values["average_precision"],
    })
    return point, details

def _resample_identity_indices(
    labels: Sequence[str],
    generator: np.random.Generator,
) -> np.ndarray:
    labels_array = np.asarray(labels).astype(str)
    identities = np.unique(labels_array)
    if not len(identities):
        return np.empty(0, dtype=np.int64)
    selected = generator.choice(identities, size=len(identities), replace=True)
    return np.concatenate([np.flatnonzero(labels_array == identity) for identity in selected])

def bootstrap_closed_details(
    details: pd.DataFrame,
    *,
    replicates: int,
    seed: int,
) -> dict[str, np.ndarray]:
    valid = details.loc[details["valid"].astype(bool)].reset_index(drop=True)
    generator = np.random.default_rng(seed)
    output = {"Rank-1": np.empty(replicates), "mAP": np.empty(replicates)}
    for replicate in range(replicates):
        indices = _resample_identity_indices(valid["identity"], generator)
        output["Rank-1"][replicate] = float(valid["rank1"].to_numpy()[indices].mean())
        output["mAP"][replicate] = float(valid["average_precision"].to_numpy()[indices].mean())
    return output

def save_bootstrap(path: Path, values: Mapping[str, np.ndarray], metadata: Mapping[str, Any]) -> None:
    arrays = {f"metric__{key}": np.asarray(value, dtype=np.float64) for key, value in values.items()}
    atomic_save_npz(path, metadata_json=np.asarray(stable_json(dict(metadata))), **arrays)

def load_bootstrap(path: Path) -> tuple[dict[str, np.ndarray], dict[str, Any]]:
    with np.load(path, allow_pickle=False) as cached:
        metadata = json.loads(str(cached["metadata_json"].item()))
        values = {
            key.removeprefix("metric__"): np.asarray(cached[key], dtype=float)
            for key in cached.files
            if key.startswith("metric__")
        }
    return values, metadata

def verified_bootstrap(
    path: Path,
    *,
    expected_metadata: Mapping[str, Any],
    expected_metrics: Sequence[str],
    expected_replicates: int,
    compute: Callable[[], Mapping[str, np.ndarray]],
) -> tuple[dict[str, np.ndarray], bool]:
    path = Path(path)
    if path.is_file():
        try:
            values, metadata = load_bootstrap(path)
            if stable_json(metadata) != stable_json(dict(expected_metadata)):
                raise ValueError("bootstrap metadata changed")
            if set(values) != set(expected_metrics):
                raise ValueError("bootstrap metric set changed")
            if any(len(values[metric]) != int(expected_replicates) for metric in expected_metrics):
                raise ValueError("bootstrap replicate count changed")
            return values, True
        except Exception as exc:
            print(f"BOOTSTRAP CACHE MISS â€” {path}: {exc}")
    values = {key: np.asarray(value, dtype=float) for key, value in compute().items()}
    save_bootstrap(path, values, expected_metadata)
    return values, False

def hierarchical_bootstrap_summary(
    point_rows: pd.DataFrame,
    *,
    metrics: Sequence[str],
    bootstrap_path_column: str,
    group_columns: Sequence[str],
    replicates: int,
    seed: int,
) -> pd.DataFrame:
    rows = []
    grouper: Any = list(group_columns)
    for group_key, group in point_rows.groupby(grouper, sort=True):
        if not isinstance(group_key, tuple):
            group_key = (group_key,)
        loaded = [load_bootstrap(Path(path))[0] for path in group[bootstrap_path_column]]
        generator = np.random.default_rng(stable_seed(seed, *group_key, "hierarchical"))
        for metric in metrics:
            distribution = np.empty(replicates)
            for replicate in range(replicates):
                selected_splits = generator.integers(0, len(loaded), size=len(loaded))
                samples = []
                for split_index in selected_splits:
                    values = loaded[int(split_index)][metric]
                    samples.append(values[generator.integers(0, len(values))])
                distribution[replicate] = float(np.nanmean(samples))
            point = float(group[metric].mean())
            rows.append(
                {
                    **dict(zip(group_columns, group_key)),
                    "metric": metric,
                    "mean": point,
                    "ci_low": float(np.nanpercentile(distribution, 2.5)),
                    "ci_high": float(np.nanpercentile(distribution, 97.5)),
                    "n_splits": int(len(group)),
                    "bootstrap_replicates": int(replicates),
                    "bootstrap_method": "split-seed bootstrap + within-split identity-block bootstrap",
                }
            )
    return pd.DataFrame(rows)

def paired_closed_bootstrap_difference(
    reference: pd.DataFrame,
    comparison: pd.DataFrame,
    *,
    replicates: int,
    seed: int,
) -> dict[str, np.ndarray]:
    required = {"image_id", "identity", "valid", "rank1", "average_precision"}
    if not required <= set(reference) or not required <= set(comparison):
        raise ValueError("Paired closed-set details are missing required columns")
    ref = reference.sort_values("image_id", kind="mergesort").reset_index(drop=True)
    cmp = comparison.sort_values("image_id", kind="mergesort").reset_index(drop=True)
    for column in ("image_id", "identity", "valid"):
        if not np.array_equal(ref[column].to_numpy(), cmp[column].to_numpy()):
            raise ValueError(f"Paired closed-set details do not align for {column}")
    valid = ref["valid"].astype(bool).to_numpy()
    ref = ref.loc[valid].reset_index(drop=True)
    cmp = cmp.loc[valid].reset_index(drop=True)
    generator = np.random.default_rng(seed)
    output = {"Rank-1": np.empty(replicates), "mAP": np.empty(replicates)}
    for replicate in range(replicates):
        indices = _resample_identity_indices(ref["identity"], generator)
        output["Rank-1"][replicate] = float(
            ref["rank1"].to_numpy()[indices].mean() - cmp["rank1"].to_numpy()[indices].mean()
        )
        output["mAP"][replicate] = float(
            ref["average_precision"].to_numpy()[indices].mean()
            - cmp["average_precision"].to_numpy()[indices].mean()
        )
    return output

def paired_difference_summary(
    *,
    bootstrap_paths: Sequence[Path],
    point_differences: Mapping[str, float],
    metrics: Sequence[str],
    replicates: int,
    seed: int,
    metadata: Mapping[str, Any],
) -> pd.DataFrame:
    loaded = [load_bootstrap(path)[0] for path in bootstrap_paths]
    generator = np.random.default_rng(seed)
    rows = []
    for metric in metrics:
        distribution = np.empty(replicates)
        for replicate in range(replicates):
            selected_splits = generator.integers(0, len(loaded), size=len(loaded))
            distribution[replicate] = float(
                np.mean(
                    [
                        loaded[int(split_index)][metric][
                            generator.integers(0, len(loaded[int(split_index)][metric]))
                        ]
                        for split_index in selected_splits
                    ]
                )
            )
        low = float(np.percentile(distribution, 2.5))
        high = float(np.percentile(distribution, 97.5))
        rows.append(
            {
                **dict(metadata),
                "metric": metric,
                "mean_difference_reference_minus_comparison": float(point_differences[metric]),
                "ci_low": low,
                "ci_high": high,
                "paired_95ci_excludes_zero": bool(low > 0.0 or high < 0.0),
                "bootstrap_replicates": int(replicates),
                "bootstrap_method": "paired split-seed + identity-block bootstrap",
            }
        )
    return pd.DataFrame(rows)

def dataset_macro_hierarchical_bootstrap(
    point_rows: pd.DataFrame,
    *,
    task: str,
    metrics: Sequence[str],
    bootstrap_path_column: str,
    replicates: int,
    seed: int,
) -> pd.DataFrame:
    required = {"model", "dataset", bootstrap_path_column, *metrics}
    if not required <= set(point_rows):
        raise ValueError(f"Dataset-macro input missing {sorted(required - set(point_rows))}")
    rows = []
    for model, model_rows in point_rows.groupby("model", sort=True):
        datasets = sorted(model_rows["dataset"].astype(str).unique())
        loaded: dict[str, list[dict[str, np.ndarray]]] = {
            dataset: [
                load_bootstrap(Path(path))[0]
                for path in model_rows.loc[
                    model_rows["dataset"].astype(str) == dataset, bootstrap_path_column
                ]
            ]
            for dataset in datasets
        }
        generator = np.random.default_rng(stable_seed(seed, task, model, "dataset-macro"))
        for metric in metrics:
            distribution = np.empty(replicates)
            for replicate in range(replicates):
                chosen_datasets = generator.choice(datasets, size=len(datasets), replace=True)
                dataset_values = []
                for dataset in chosen_datasets:
                    split_values = loaded[str(dataset)]
                    split = split_values[generator.integers(0, len(split_values))]
                    values = split[metric]
                    dataset_values.append(values[generator.integers(0, len(values))])
                distribution[replicate] = float(np.nanmean(dataset_values))
            point = float(
                model_rows.groupby("dataset", sort=True)[metric].mean().mean()
            )
            rows.append(
                {
                    "task": task,
                    "model": model,
                    "metric": metric,
                    "aggregation": "Dataset-Macro Average",
                    "mean": point,
                    "ci_low": float(np.nanpercentile(distribution, 2.5)),
                    "ci_high": float(np.nanpercentile(distribution, 97.5)),
                    "number_datasets": len(datasets),
                    "bootstrap_replicates": int(replicates),
                    "bootstrap_method": "dataset + split-seed + identity-block hierarchical bootstrap",
                }
            )
    return pd.DataFrame(rows)

In [19]:
"""CPU diagnostics for frozen, closed-set identity embeddings.

All sampling depends on metadata and a common seed, never model scores.
ROC operating points are descriptive test-curve summaries, not fitted deployment
thresholds. Missing species are never replaced by dataset names.
"""
from __future__ import annotations

import hashlib
import json
from collections import Counter
import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import linkage, cut_tree
from sklearn.metrics import adjusted_rand_score, roc_auc_score, roc_curve, silhouette_samples
from sklearn.metrics.cluster import contingency_matrix


REPRESENTATION_VERSION = "closed-representation-1.0.0"
PAIR_GROUPS = ("positive", "negative_all", "negative_same_species", "negative_cross_species")


def rep_seed(*parts):
    return int.from_bytes(hashlib.sha256(json.dumps(parts, default=str).encode()).digest()[:8], "big") % (2**32-1)


def rep_normalize(embeddings):
    z = np.asarray(embeddings, dtype=np.float64)
    if z.ndim != 2 or not np.isfinite(z).all():
        raise ValueError("Expected a finite [images, dimensions] embedding matrix")
    norms = np.linalg.norm(z, axis=1, keepdims=True)
    if np.any(norms <= 1e-12):
        raise ValueError("Zero-norm embeddings are not supported")
    return z / norms


def rep_metadata(frame, species_columns, group_columns, dataset_species):
    f = frame.copy().reset_index(drop=True)
    for column in ("image_id", "identity", "dataset"):
        if column not in f or f[column].fillna("").astype(str).str.strip().eq("").any():
            raise ValueError(f"Missing or empty {column} metadata")
        f[column] = f[column].astype(str)
    if f.image_id.duplicated().any():
        raise ValueError("Diagnostic image IDs must be unique")
    species = pd.Series("", index=f.index, dtype=object)
    source = pd.Series("missing", index=f.index, dtype=object)
    missing_labels = {"", "unknown", "unk", "none", "nan", "null", "n/a"}
    for column in species_columns:
        if column in f:
            values = f[column].fillna("").astype(str).str.strip()
            values.loc[values.str.lower().isin(missing_labels)] = ""
            conflict = species.ne("") & values.ne("") & species.ne(values)
            if conflict.any():
                raise ValueError(f"Conflicting explicit species columns: {column}")
            use = species.eq("") & values.ne("")
            species.loc[use], source.loc[use] = values.loc[use], column
    for dataset, value in dataset_species.items():
        value = str(value).strip()
        if value.lower() in missing_labels:
            raise ValueError("Dataset species overrides must be nonempty verified labels")
        use = f.dataset.eq(dataset)
        if (use & species.ne("") & species.ne(value)).any():
            raise ValueError(f"Species override conflicts with explicit metadata for {dataset}")
        missing = use & species.eq("")
        species.loc[missing], source.loc[missing] = value, "verified_dataset_override"
    f["rep_species"], f["rep_species_source"] = species, source
    for _, group in f.groupby(["dataset", "identity"], sort=False):
        labels = set(group.rep_species) - {""}
        if len(labels) > 1:
            raise ValueError("A single identity has conflicting species labels")
        if len(labels) == 1:
            missing = group.index[group.rep_species.eq("")]
            f.loc[missing, "rep_species"] = next(iter(labels))
            f.loc[missing, "rep_species_source"] = "same_identity_explicit_label"
    f["rep_observation_group"] = ""
    f["rep_group_source"] = "missing"
    # Select one declared group column; coalescing different group namespaces
    # would create spurious agreements or disagreements.
    for column in group_columns:
        if column in f and f[column].fillna("").astype(str).str.strip().ne("").any():
            values = f[column].fillna("").astype(str).str.strip()
            values.loc[values.str.lower().isin(missing_labels)] = ""
            valid = values.ne("")
            f.loc[valid, "rep_observation_group"] = [json.dumps([d, v]) for d, v in zip(f.loc[valid, "dataset"], values.loc[valid])]
            f.loc[valid, "rep_group_source"] = column
            break
    return f


def rep_balanced_indices(frame, per_identity, maximum, seed):
    """Round-robin identities, then sample images; ordering is cache-order invariant."""
    if per_identity < 1 or maximum < 1:
        raise ValueError("Sampling limits must be positive")
    f = frame.reset_index(drop=True)
    rng = np.random.default_rng(seed)
    groups = []
    for identity in sorted(f.identity.unique()):
        indices = f.index[f.identity.eq(identity)].to_numpy()
        indices = indices[np.argsort(f.loc[indices, "image_id"].to_numpy(), kind="stable")]
        groups.append(rng.permutation(indices)[:per_identity].tolist())
    groups = [groups[i] for i in rng.permutation(len(groups))]
    chosen = []
    for level in range(per_identity):
        for group in groups:
            if level < len(group):
                chosen.append(group[level])
                if len(chosen) == maximum:
                    return np.asarray(chosen, dtype=int)
    return np.asarray(chosen, dtype=int)


def rep_group_weights(frame, group_columns=("identity", "query_id")):
    """Equal identity, equal query within identity, equal pair within query."""
    if frame.empty:
        return np.empty(0, dtype=float)
    identity, query = group_columns
    number_ids = frame[identity].nunique()
    nq = frame.groupby(identity)[query].transform("nunique").to_numpy(float)
    npairs = frame.groupby([identity, query])[query].transform("size").to_numpy(float)
    return 1.0 / (number_ids * nq * npairs)


def rep_weighted_quantile(values, weights, quantile):
    v, w = np.asarray(values, float), np.asarray(weights, float)
    valid = np.isfinite(v) & np.isfinite(w) & (w > 0)
    if not valid.any():
        return float("nan")
    order = np.argsort(v[valid], kind="stable")
    v, w = v[valid][order], w[valid][order]
    cdf = np.cumsum(w) / w.sum()
    return float(v[min(np.searchsorted(cdf, quantile, side="left"), len(v)-1)])


def rep_pair_roc(pairs, negative_group, far_targets):
    pos = pairs.loc[pairs.pair_group.eq("positive")].copy()
    neg = pairs.loc[pairs.pair_group.eq(negative_group)].copy()
    shared = set(pos.query_id) & set(neg.query_id)
    pos = pos.loc[pos.query_id.isin(shared)].copy()
    neg = neg.loc[neg.query_id.isin(shared)].copy()
    result = {"AUROC": np.nan, "n_positive_pairs": len(pos), "n_negative_pairs": len(neg), "n_queries": len(shared), "status": "no_eligible_positive_and_negative_pairs"}
    for far in far_targets:
        tag = f"{far:g}"
        result.update({f"TAR_at_FAR_{tag}": np.nan, f"achieved_FAR_{tag}": np.nan, f"descriptive_ROC_threshold_{tag}": np.nan})
    if pos.empty or neg.empty:
        return result, pd.DataFrame(columns=["FAR", "TAR", "score_threshold"])
    wp, wn = rep_group_weights(pos), rep_group_weights(neg)
    scores = np.r_[pos.score.to_numpy(float), neg.score.to_numpy(float)]
    truth = np.r_[np.ones(len(pos), int), np.zeros(len(neg), int)]
    weights = np.r_[wp, wn]
    fpr, tpr, thresholds = roc_curve(truth, scores, sample_weight=weights, drop_intermediate=False)
    result["AUROC"] = float(roc_auc_score(truth, scores, sample_weight=weights))
    result["status"] = "descriptive_test_ROC_not_a_deployment_threshold"
    for far in far_targets:
        allowed = np.flatnonzero(fpr <= float(far) + 1e-12)
        # No interpolation or tie splitting: a single shared score threshold.
        index = allowed[np.argmax(tpr[allowed])]
        tag = f"{far:g}"
        result.update({f"TAR_at_FAR_{tag}": float(tpr[index]), f"achieved_FAR_{tag}": float(fpr[index]), f"descriptive_ROC_threshold_{tag}": float(thresholds[index])})
    result["minimum_observed_negative_pair_weight"] = float(wn.min() / wn.sum())
    return result, pd.DataFrame({"FAR": fpr, "TAR": tpr, "score_threshold": thresholds})


def rep_sample_candidates(indices, gallery, maximum, seed):
    if not len(indices):
        return np.empty(0, dtype=int)
    labels, images = gallery.identity.to_numpy(), gallery.image_id.to_numpy()
    indices = np.asarray(indices)[np.argsort(images[indices], kind="stable")]
    rng = np.random.default_rng(seed)
    groups = {}
    for index in rng.permutation(indices):
        groups.setdefault(labels[index], []).append(int(index))
    identities = sorted(groups)
    order = rng.permutation(len(identities))
    selected = []
    for level in range(maximum):
        for j in order:
            candidates = groups[identities[j]]
            if level < len(candidates):
                selected.append(candidates[level])
                if len(selected) == maximum:
                    return np.asarray(selected, int)
    return np.asarray(selected, int)


def rep_pair_diagnostics(query, qz, gallery, gz, config, seed):
    q, g = query.reset_index(drop=True), gallery.reset_index(drop=True)
    qz, gz = rep_normalize(qz), rep_normalize(gz)
    anchors = rep_balanced_indices(q, config["queries_per_identity"], config["max_queries"], seed)
    records, hard_rows = [], []
    gid, gimage = g.identity.to_numpy(), g.image_id.to_numpy()
    gspecies, ggroup = g.rep_species.to_numpy(), g.rep_observation_group.to_numpy()
    for start in range(0, len(anchors), config["similarity_chunk"]):
        indices = anchors[start:start+config["similarity_chunk"]]
        similarities = np.clip(qz[indices] @ gz.T, -1, 1)
        for local, index in enumerate(indices):
            row, score = q.iloc[index], similarities[local]
            eligible = gimage != row.image_id
            if row.rep_observation_group:
                eligible &= (ggroup == "") | (ggroup != row.rep_observation_group)
            positive = eligible & (gid == row.identity)
            negative = eligible & (gid != row.identity)
            same = negative & (gspecies == row.rep_species) if row.rep_species else np.zeros(len(g), bool)
            cross = negative & (gspecies != "") & (gspecies != row.rep_species) if row.rep_species else np.zeros(len(g), bool)
            row_seed = rep_seed(seed, row.image_id)
            for name, mask in zip(PAIR_GROUPS, (positive, negative, same, cross)):
                limit = config["positive_pairs_per_query"] if name == "positive" else config["negative_pairs_per_query"]
                selected = rep_sample_candidates(np.flatnonzero(mask), g, limit, rep_seed(row_seed, name))
                for candidate in selected:
                    records.append({"query_id": row.image_id, "gallery_id": gimage[candidate], "identity": row.identity, "species": row.rep_species, "pair_group": name, "score": float(score[candidate]), "both_observation_groups_available": bool(row.rep_observation_group and ggroup[candidate])})
            h = {"query_id": row.image_id, "identity": row.identity, "species": row.rep_species, "n_positive_gallery_images": int(positive.sum()), "n_negative_gallery_images": int(negative.sum()), "n_same_species_negative_gallery_images": int(same.sum()), "positive_mean_exact": float(score[positive].mean()) if positive.any() else np.nan}
            for name, mask in (("all", negative), ("same_species", same)):
                h[f"hard_margin_{name}"] = float(score[positive].mean() - score[mask].max()) if positive.any() and mask.any() else np.nan
            hard_rows.append(h)
    pairs = pd.DataFrame(records, columns=["query_id", "gallery_id", "identity", "species", "pair_group", "score", "both_observation_groups_available"])
    hard = pd.DataFrame(hard_rows)
    result = {"sampled_queries": len(anchors), "eligible_query_identities": q.iloc[anchors].identity.nunique(), "species_labeled_query_fraction": float(q.iloc[anchors].rep_species.ne("").mean()) if len(anchors) else np.nan}
    hist_rows, roc_frames, species_rows = [], [], []
    for group in PAIR_GROUPS:
        subset = pairs.loc[pairs.pair_group.eq(group)].copy()
        weights = rep_group_weights(subset)
        mean = float(np.average(subset.score, weights=weights)) if len(subset) else np.nan
        result[f"cosine_mean_{group}"] = mean
        result[f"pair_count_{group}"] = len(subset)
        result[f"pair_queries_{group}"] = subset.query_id.nunique()
        result[f"group_metadata_coverage_{group}"] = float(np.average(subset.both_observation_groups_available, weights=weights)) if len(subset) else np.nan
        for quantile in (0.1, 0.5, 0.9):
            result[f"cosine_q{int(quantile*100)}_{group}"] = rep_weighted_quantile(subset.score, weights, quantile)
        if len(subset):
            bins = np.linspace(-1, 1, 51)
            counts, _ = np.histogram(subset.score, bins=bins)
            mass, _ = np.histogram(subset.score, bins=bins, weights=weights)
            for a, b, count, prob in zip(bins[:-1], bins[1:], counts, mass):
                hist_rows.append({"pair_group": group, "bin_left": a, "bin_right": b, "pair_count": int(count), "identity_balanced_probability_mass": float(prob)})
    for label, negative in (("all", "negative_all"), ("same_species", "negative_same_species"), ("cross_species", "negative_cross_species")):
        roc, curve = rep_pair_roc(pairs, negative, config["far_targets"])
        result.update({f"verification_{label}_{key}": value for key, value in roc.items()})
        if len(curve):
            roc_frames.append(curve.assign(pair_scope=label))
    for species, subset in pairs.loc[pairs.species.ne("")].groupby("species", sort=True):
        roc, _ = rep_pair_roc(subset, "negative_same_species", config["far_targets"])
        species_rows.append({"species": species, **roc})
    result["verification_same_species_macro_AUROC"] = float(pd.Series([r["AUROC"] for r in species_rows], dtype=float).mean())
    # Compare matched eligible anchors, avoiding gaps driven by missing groups.
    def matched_group_gap(first, second):
        a = pairs.loc[pairs.pair_group.eq(first)]
        b = pairs.loc[pairs.pair_group.eq(second)]
        shared = set(a.query_id) & set(b.query_id)
        a, b = a.loc[a.query_id.isin(shared)], b.loc[b.query_id.isin(shared)]
        if a.empty or b.empty:
            return np.nan, len(shared)
        return float(np.average(a.score, weights=rep_group_weights(a)) - np.average(b.score, weights=rep_group_weights(b))), len(shared)
    for name, a, b in (("identity_gap_all", "positive", "negative_all"), ("identity_gap_same_species", "positive", "negative_same_species"), ("species_gap", "negative_same_species", "negative_cross_species")):
        result[name], result[name+"_matched_queries"] = matched_group_gap(a, b)
    result["alignment_squared_l2"] = float(2-2*result["cosine_mean_positive"])
    for scope in ("all", "same_species"):
        selected = hard.loc[np.isfinite(hard[f"hard_margin_{scope}"])].copy() if len(hard) else pd.DataFrame()
        result[f"hard_margin_{scope}_eligible_queries"] = len(selected)
        for name in ("mean", "median", "p10", "violation_rate"):
            result[f"hard_margin_{scope}_{name}"] = np.nan
        if len(selected):
            weights = rep_group_weights(selected)
            values = selected[f"hard_margin_{scope}"].to_numpy()
            result[f"hard_margin_{scope}_mean"] = float(np.average(values, weights=weights))
            result[f"hard_margin_{scope}_median"] = rep_weighted_quantile(values, weights, 0.5)
            result[f"hard_margin_{scope}_p10"] = rep_weighted_quantile(values, weights, 0.1)
            result[f"hard_margin_{scope}_violation_rate"] = float(np.average(values <= 0, weights=weights))
    fingerprint = hashlib.sha256(pairs[["query_id", "gallery_id", "pair_group"]].to_csv(index=False).encode()).hexdigest()
    result["pair_membership_sha256"] = fingerprint
    return result, hard, pairs, pd.DataFrame(hist_rows), pd.concat(roc_frames, ignore_index=True) if roc_frames else pd.DataFrame(columns=["FAR", "TAR", "score_threshold", "pair_scope"]), pd.DataFrame(species_rows)


def rep_effective_rank(z, centered=False):
    values = rep_normalize(z)
    if centered:
        values = values - values.mean(axis=0, keepdims=True)
    gram = values @ values.T if len(values) <= values.shape[1] else values.T @ values
    eigenvalues = np.linalg.eigvalsh(gram)
    cutoff = max(float(np.max(eigenvalues)), 1.0) * 1e-12
    singular_values = np.sqrt(eigenvalues[eigenvalues > cutoff])
    if not len(singular_values):
        return 0.0
    p = singular_values / singular_values.sum()
    return float(np.exp(-np.sum(p * np.log(p))))


def rep_b3(true_labels, predicted_labels):
    counts = contingency_matrix(true_labels, predicted_labels).astype(float)
    n = counts.sum()
    precision = float((counts**2 / counts.sum(axis=0, keepdims=True)).sum() / n)
    recall = float((counts**2 / counts.sum(axis=1, keepdims=True)).sum() / n)
    return precision, recall, float(2*precision*recall/(precision+recall))


def rep_geometry(frame, embeddings, config, seed):
    chosen = rep_balanced_indices(frame, config["geometry_per_identity"], config["geometry_max_samples"], seed)
    f, z = frame.iloc[chosen].reset_index(drop=True), rep_normalize(embeddings)[chosen]
    n, dim = z.shape
    ids = f.identity.to_numpy()
    species = f.rep_species.to_numpy()
    k = len(set(ids))
    cosine = np.clip(z @ z.T, -1, 1)
    distance = np.maximum(1-cosine, 0)
    np.fill_diagonal(distance, 0)
    result = {"geometry_samples": n, "geometry_identities": k, "geometry_dimension": dim, "geometry_membership_sha256": hashlib.sha256(f.image_id.to_csv(index=False).encode()).hexdigest(), "effective_rank_uncentered": rep_effective_rank(z), "effective_rank_centered": rep_effective_rank(z, centered=True), "effective_rank_maximum_uncentered": min(n, dim), "effective_rank_maximum_centered": min(max(n-1, 0), dim), "uniformity_t": config["uniformity_t"]}
    result["effective_rank_uncentered_fraction"] = result["effective_rank_uncentered"] / min(n, dim)
    result["effective_rank_centered_fraction"] = result["effective_rank_centered"] / min(n-1, dim) if n > 1 else np.nan
    triangle = np.triu_indices(n, 1)
    result["uniformity_log_mean_exp"] = float(np.log(np.exp(-2*config["uniformity_t"]*distance[triangle]).mean())) if n > 1 else np.nan
    detail = f[["image_id", "identity", "dataset", "rep_species"]].copy()
    detail["silhouette_identity_all"] = np.nan
    detail["silhouette_identity_same_species"] = np.nan
    result["silhouette_identity_all"] = np.nan
    result["silhouette_identity_same_species"] = np.nan
    result["silhouette_species"] = np.nan
    counts = Counter(ids)
    eligible = np.asarray([counts[i] >= 2 for i in ids])
    result["silhouette_nonsingleton_samples"] = int(eligible.sum())
    if 2 <= k < n:
        scores = silhouette_samples(distance, ids, metric="precomputed")
        detail.loc[eligible, "silhouette_identity_all"] = scores[eligible]
        result["silhouette_identity_all"] = float(detail.groupby("identity").silhouette_identity_all.mean().mean())
    for label in sorted(set(species)-{""}):
        ix = np.flatnonzero(species == label)
        labels = ids[ix]
        if 2 <= len(set(labels)) < len(ix):
            scores = silhouette_samples(distance[np.ix_(ix, ix)], labels, metric="precomputed")
            c = Counter(labels)
            good = np.asarray([c[a] >= 2 for a in labels])
            detail.loc[ix[good], "silhouette_identity_same_species"] = scores[good]
    by_species_id = detail.groupby(["rep_species", "identity"]).silhouette_identity_same_species.mean()
    result["silhouette_identity_same_species"] = float(by_species_id.groupby(level=0).mean().mean())
    result["silhouette_same_species_eligible_samples"] = int(detail.silhouette_identity_same_species.notna().sum())
    ix = np.flatnonzero(species != "")
    if len(ix) and 2 <= len(set(species[ix])) < len(ix):
        scores = silhouette_samples(distance[np.ix_(ix, ix)], species[ix], metric="precomputed")
        counts_species = Counter(species[ix])
        valid = np.asarray([counts_species[s] >= 2 for s in species[ix]])
        tmp = pd.DataFrame({"species": species[ix][valid], "score": scores[valid]})
        result["silhouette_species"] = float(tmp.groupby("species").score.mean().mean())
    result.update({"closed_clustering_B3_precision_known_K": np.nan, "closed_clustering_B3_recall_known_K": np.nan, "closed_clustering_B3_F1_known_K": np.nan, "closed_clustering_ARI_known_K": np.nan, "closed_clustering_K": k, "closed_clustering_status": "disabled" if not config["known_k_clustering"] else "insufficient_repeated_identities"})
    if config["known_k_clustering"] and 2 <= k < n:
        # Conditional closed-set diagnostic: K is the known identity count in
        # this label-balanced sample, never estimated from a novel population.
        tree = linkage(distance[triangle], method="average")
        clusters = cut_tree(tree, n_clusters=[k]).reshape(-1)
        precision, recall, f1 = rep_b3(ids, clusters)
        result.update({"closed_clustering_B3_precision_known_K": precision, "closed_clustering_B3_recall_known_K": recall, "closed_clustering_B3_F1_known_K": f1, "closed_clustering_ARI_known_K": float(adjusted_rand_score(ids, clusters)), "closed_clustering_status": "conditional_known_K_balanced_sample"})
        detail["closed_cluster_label_known_K"] = clusters
    return result, detail, chosen


REP_METRICS = {
    "verification_all_AUROC": ("higher", "0_to_1"),
    "verification_same_species_AUROC": ("higher", "0_to_1"),
    "verification_same_species_macro_AUROC": ("higher", "0_to_1"),
    "cosine_mean_positive": ("diagnostic", "cosine"),
    "cosine_mean_negative_all": ("diagnostic", "cosine"),
    "cosine_mean_negative_same_species": ("diagnostic", "cosine"),
    "cosine_mean_negative_cross_species": ("diagnostic", "cosine"),
    "identity_gap_all": ("higher", "cosine_difference"),
    "identity_gap_same_species": ("higher", "cosine_difference"),
    "species_gap": ("diagnostic", "cosine_difference"),
    "alignment_squared_l2": ("lower_with_separation", "squared_unit_distance"),
    "silhouette_identity_all": ("higher", "minus_1_to_1"),
    "silhouette_identity_same_species": ("higher", "minus_1_to_1"),
    "silhouette_species": ("diagnostic", "minus_1_to_1"),
    "effective_rank_uncentered": ("diagnostic", "rank"),
    "effective_rank_centered": ("diagnostic", "rank"),
    "effective_rank_uncentered_fraction": ("diagnostic", "0_to_1"),
    "effective_rank_centered_fraction": ("diagnostic", "0_to_1"),
    "uniformity_log_mean_exp": ("diagnostic", "log_expectation"),
    "closed_clustering_B3_precision_known_K": ("higher", "0_to_1"),
    "closed_clustering_B3_recall_known_K": ("higher", "0_to_1"),
    "closed_clustering_B3_F1_known_K": ("higher", "0_to_1"),
    "closed_clustering_ARI_known_K": ("higher", "ARI"),
}
for _scope in ("all", "same_species"):
    for _stat in ("mean", "median", "p10", "violation_rate"):
        REP_METRICS[f"hard_margin_{_scope}_{_stat}"] = ("lower" if _stat == "violation_rate" else "higher", "0_to_1" if _stat == "violation_rate" else "cosine_difference")


def rep_metric_registry(far_targets):
    result = dict(REP_METRICS)
    for scope in ("all", "same_species"):
        for far in far_targets:
            result[f"verification_{scope}_TAR_at_FAR_{far:g}"] = ("higher", "0_to_1")
            result[f"verification_{scope}_achieved_FAR_{far:g}"] = ("diagnostic", "0_to_1")
    return result


def rep_summaries(results, metric_registry, expected_splits, expected_datasets):
    """Descriptive split SD, not confidence intervals from overlapping splits."""
    rows = []
    for (model, dataset), group in results.groupby(["model", "dataset"], sort=True):
        for metric, (direction, unit) in metric_registry.items():
            values = pd.to_numeric(group.get(metric, pd.Series(np.nan, index=group.index)), errors="coerce")
            valid = values[np.isfinite(values)]
            rows.append({"model": model, "dataset": dataset, "metric": metric, "mean": valid.mean(), "split_sd": valid.std(ddof=1), "min": valid.min(), "max": valid.max(), "n_valid_splits": len(valid), "n_expected_splits": expected_splits, "complete": len(valid) == expected_splits, "direction": direction, "unit": unit, "uncertainty": "descriptive split SD; no inferential CI"})
    summary = pd.DataFrame(rows)
    macro_rows = []
    for (model, metric), group in summary.groupby(["model", "metric"], sort=True):
        eligible = group.loc[group.complete]
        complete = len(eligible) == len(expected_datasets) and set(eligible.dataset) == set(expected_datasets)
        macro_rows.append({"model": model, "metric": metric, "mean": float(eligible["mean"].mean()) if complete else np.nan, "available_dataset_mean": float(eligible["mean"].mean()) if len(eligible) else np.nan, "n_complete_datasets": len(eligible), "n_expected_datasets": len(expected_datasets), "complete": complete, "datasets_in_available_mean": "|".join(sorted(eligible.dataset)), "direction": group.direction.iloc[0], "unit": group.unit.iloc[0], "uncertainty": "no inferential CI; headline requires complete coverage"})
    return summary, pd.DataFrame(macro_rows)


## Validate frozen inputs and resolve model names

In [20]:
STAGE00_MANIFEST_PATH = STAGE00_ROOT / "preparation_manifest.json"
STAGE01_MANIFEST_PATH = STAGE01_ROOT / "stage01_manifest.json"
stage00_manifest = json.loads(STAGE00_MANIFEST_PATH.read_text())
stage01_manifest = json.loads(STAGE01_MANIFEST_PATH.read_text())
if stage00_manifest.get("status") != "passed" or not stage00_manifest.get("apexid_split_v2", {}).get("passed"):
    raise RuntimeError("Stage 00 or its Split V2 integrity record is not marked passed")
if stage01_manifest.get("status") != "passed":
    raise RuntimeError("Stage 01 is not marked passed")
available_models = stage01_manifest.get("models", {})
completed = tuple(stage01_manifest.get("models_completed", ()))
MODEL_NAMES = tuple(MODELS_TO_EVALUATE) if MODELS_TO_EVALUATE is not None else completed
if len(MODEL_NAMES) < 1 or len(set(MODEL_NAMES)) != len(MODEL_NAMES):
    raise ValueError("Choose at least one distinct model from Stage 01")
if not set(MODEL_NAMES) <= set(available_models) or not set(MODEL_NAMES) <= set(completed):
    missing = sorted(set(MODEL_NAMES) - set(available_models.keys()) | set(MODEL_NAMES) - set(completed))
    raise ValueError(
        "Requested models missing from Stage 01 manifest.\n"
        f"  requested: {list(MODEL_NAMES)}\n"
        f"  models_completed: {sorted(completed)}\n"
        f"  missing: {missing}\n"
        "Set MODELS_TO_EVALUATE or APEXID_STAGE02_MODELS to a folder name under embeddings/."
    )
EXPECTED_MODELS = set(MODEL_NAMES)
model_metadata_by_name = {name: available_models[name] for name in MODEL_NAMES}
if (STAGE01_ROOT / "embeddings").is_dir():
    EMBEDDINGS_ROOT = STAGE01_ROOT / "embeddings"
else:
    source = stage01_manifest.get("embedding_source", {}).get("root")
    if not source:
        raise FileNotFoundError("Stage 01 contains neither embeddings/ nor an embedding_source.root")
    EMBEDDINGS_ROOT = Path(source)
    if not EMBEDDINGS_ROOT.is_absolute():
        EMBEDDINGS_ROOT = STAGE01_ROOT / EMBEDDINGS_ROOT
print("Stage 01 models_completed:", sorted(completed))
print("EMBEDDINGS_ROOT:", EMBEDDINGS_ROOT)
dataset_manifest_hashes = {d: stage00_manifest["dataset_manifest_hashes"][d] for d in EXTERNAL_DATASETS}
canonical_frames, split_manifests, split_manifest_hashes = {}, {}, {}
observed_source_hashes, embedding_cache_paths, embedding_file_hashes = {}, {}, {}
for dataset in EXTERNAL_DATASETS:
    path = STAGE00_ROOT / "manifests" / f"{dataset}_canonical_manifest.csv"
    frame = pd.read_csv(path, dtype=str).fillna("")
    if not {"image_id", "identity", "dataset", "path"} <= set(frame):
        raise ValueError(f"Incomplete canonical manifest: {dataset}")
    if frame.image_id.duplicated().any() or set(frame.dataset) != {dataset}:
        raise ValueError(f"Invalid canonical membership: {dataset}")
    canonical_frames[dataset] = frame
    observed_source_hashes[str(path)] = sha256_file(path)
    for split_seed in SPLIT_SEEDS:
        path = STAGE00_ROOT / "splits" / dataset / f"seed_{split_seed}" / "split_manifest.csv"
        split = pd.read_csv(path, dtype=str).fillna("")
        if split.image_id.duplicated().any() or set(split.image_id) != set(frame.image_id):
            raise ValueError(f"Invalid frozen split membership: {dataset}/{split_seed}")
        if not {"image_id", "identity", "closed_role"} <= set(split):
            raise ValueError("Frozen split lacks closed_role or identity")
        expected = frame.set_index("image_id").loc[split.image_id, "identity"].to_numpy()
        if not np.array_equal(expected, split.identity.to_numpy()):
            raise ValueError("Canonical and split identity labels disagree")
        gallery = split.loc[split.closed_role.eq("gallery")]
        query = split.loc[split.closed_role.eq("query_known")]
        if gallery.empty or query.empty or not set(query.identity) <= set(gallery.identity):
            raise ValueError("Closed-set queries must have gallery identities and both partitions must be nonempty")
        split_manifests[(dataset, split_seed)] = split
        split_manifest_hashes[(dataset, split_seed)] = stage00_manifest["split_manifest_hashes"][f"{dataset}::seed_{split_seed}"]
        observed_source_hashes[str(path)] = sha256_file(path)
        for filename in ("split_record.json", "split_diagnostics.json"):
            if not (path.parent / filename).is_file():
                raise FileNotFoundError(path.parent / filename)
    for model in MODEL_NAMES:
        path = EMBEDDINGS_ROOT / model / dataset / "embeddings.npz"
        if not path.is_file():
            raise FileNotFoundError(
                f"Embedding cache missing: {path}\n"
                f"Expected: {EMBEDDINGS_ROOT}/<model>/<dataset>/embeddings.npz"
            )
        with np.load(path, allow_pickle=False) as saved:
            meta = json.loads(str(saved["metadata_json"].item()))
        if meta.get("model") != model or meta.get("dataset") != dataset or meta.get("dataset_manifest_sha256") != dataset_manifest_hashes[dataset]:
            raise ValueError(f"Embedding provenance mismatch: {path}")
        embedding_cache_paths[(model, dataset)] = path
        embedding_file_hashes[(model, dataset)] = sha256_file(path)
inventory = pd.read_csv(STAGE01_ROOT / "embedding_inventory.csv")
expected_pairs = {(m, d) for m in MODEL_NAMES for d in EXTERNAL_DATASETS}
inventory_pairs = set(zip(inventory.model.astype(str), inventory.dataset.astype(str)))
if not expected_pairs <= inventory_pairs:
    missing_pairs = sorted(expected_pairs - inventory_pairs)
    raise ValueError(
        f"embedding_inventory.csv missing {len(missing_pairs)} row(s), e.g. {missing_pairs[:8]}"
    )
checks = pd.read_csv(STAGE00_ROOT / "tables" / "split_integrity_checks.csv")
checks = checks.loc[checks.dataset.isin(EXTERNAL_DATASETS)]
if len(checks) != len(EXTERNAL_DATASETS)*len(SPLIT_SEEDS) or not checks.passed.astype(str).str.lower().eq("true").all():
    raise ValueError("Frozen split integrity checks failed or are incomplete")
model_summary = pd.read_csv(STAGE01_ROOT / "tables" / "model_summary.csv")
model_summary = model_summary.loc[model_summary.model.isin(MODEL_NAMES)].copy()
if set(model_summary.model) != EXPECTED_MODELS:
    raise ValueError(
        "model_summary.csv missing models "
        f"{sorted(EXPECTED_MODELS - set(model_summary.model))}; "
        f"present: {sorted(model_summary.model.unique())}"
    )
PATHS = {name: OUTPUT_ROOT / name for name in ("tables", "closed_set", "scores", "statistics", "representation", "plots", "reports")}
for path in PATHS.values():
    path.mkdir(parents=True, exist_ok=True)
runtime_recorder = RuntimeRecorder()
print("Closed-set models:", MODEL_NAMES)
print("Frozen datasets:", EXTERNAL_DATASETS)
print("Output:", OUTPUT_ROOT)
display(model_summary)


Stage 01 models_completed: ['ApexID_Convnext_Reweighted']
EMBEDDINGS_ROOT: /kaggle/input/datasets/mohamedbassat/apexid-cvpr-stage01-bhca-logistic/apexid_cvpr_stage01_bhca_logistic/embeddings
Closed-set models: ('ApexID_Convnext_Reweighted',)
Frozen datasets: ('bristol_gorillas', 'elephants', 'gcn_id', 'jaguar_antonio')
Output: /kaggle/working/apexid_cvpr_closed_representation


,model,model_identifier,checkpoint_path,checkpoint_sha256,config_path,config_sha256,config_provenance,revision,embedding_dimension,input_resolution,parameters,l2_normalized,transform,mixed_precision_validation,selected_batch_size,checkpoint_provenance_type
0,ApexID_Convnext_Reweighted,/kaggle/input/models/mohamedbassat/apexid-conv...,/kaggle/input/models/mohamedbassat/apexid-conv...,936d87678685daaac1948ffd54b02aedf4b050f75af614...,/kaggle/input/models/mohamedbassat/apexid-conv...,8dd01225f75af78a0dfba4c611fd0023622980cfa6f140...,sidecar YAML next to best_backbone.pt,149,2048,"[440,440]",unknown,True,"{""config_sha256"":""8dd01225f75af78a0dfba4c611fd...","{""extraction_device"":""cuda_if_available"",""sour...",64,bhca_backbone_sidecar


## Align cached embeddings and metadata

In [21]:
def canonical_embeddings_for(model_name, dataset_name):
    path = embedding_cache_paths[(model_name, dataset_name)]
    embeddings, metadata, image_ids, stored_paths, stored_ids = load_embedding_cache(path)
    canonical = canonical_frames[dataset_name]
    if len(set(image_ids)) != len(image_ids) or len(image_ids) != len(canonical) or set(image_ids) != set(canonical.image_id):
        raise ValueError(f"Embedding membership mismatch: {path}")
    frame = canonical.set_index("image_id", drop=False).loc[image_ids].reset_index(drop=True)
    if not np.array_equal(stored_paths, frame.path.to_numpy()) or not np.array_equal(stored_ids, frame.identity.to_numpy()):
        raise ValueError("Embedding paths or identities do not match canonical metadata")
    # Metadata may also be retained only in the split manifest. It must agree
    # across all splits; role assignments are not copied into canonical data.
    for column in (*SPECIES_COLUMNS, *OBSERVATION_GROUP_COLUMNS):
        if column == "cache_species":
            continue
        for seed in SPLIT_SEEDS:
            split = split_manifests[(dataset_name, seed)]
            if column not in split:
                continue
            values = split.set_index("image_id").loc[frame.image_id, column].fillna("").astype(str).to_numpy()
            if column not in frame:
                frame[column] = values
            else:
                existing = frame[column].fillna("").astype(str).to_numpy()
                conflict = (existing != "") & (values != "") & (existing != values)
                if conflict.any():
                    raise ValueError(f"Conflicting metadata {column}: {dataset_name}")
                frame.loc[existing == "", column] = values[existing == ""]
    with np.load(path, allow_pickle=False) as saved:
        if "species" in saved.files:
            values = saved["species"].astype(str)
            if values.ndim != 1 or len(values) != len(frame):
                raise ValueError("Embedding species array is not aligned")
            frame["cache_species"] = values
    return rep_metadata(frame, SPECIES_COLUMNS, OBSERVATION_GROUP_COLUMNS, VERIFIED_DATASET_SPECIES), embeddings, metadata


def rows_for_roles(frame, embeddings, split_manifest, role_column, roles):
    selected = split_manifest.loc[split_manifest[role_column].isin(roles)].reset_index(drop=True)
    positions = pd.Series(np.arange(len(frame)), index=frame.image_id)
    indices = positions.loc[selected.image_id].to_numpy(int)
    enriched = frame.iloc[indices].reset_index(drop=True).copy()
    if not np.array_equal(enriched.identity, selected.identity):
        raise ValueError("Selected identity alignment mismatch")
    return enriched, embeddings[indices]


def run_representation_cached(model, dataset, split_seed, q, qz, g, gz, embedding_hash):
    directory = PATHS["representation"] / model / dataset / f"seed_{split_seed}"
    directory.mkdir(parents=True, exist_ok=True)
    metadata_columns = ["image_id", "identity", "dataset", "rep_species", "rep_observation_group"]
    provenance = {"version": REPRESENTATION_VERSION, "implementation_sha256": REP_IMPLEMENTATION_SHA256, "model": model, "dataset": dataset, "split_seed": split_seed, "config": REP_CONFIG, "export_pair_scores": EXPORT_PAIR_SCORES, "embedding_sha256": embedding_hash, "query_metadata_sha256": sha256_bytes(q[metadata_columns].to_csv(index=False).encode()), "gallery_metadata_sha256": sha256_bytes(g[metadata_columns].to_csv(index=False).encode())}
    point_path = directory / "metrics.json"
    required = [directory / name for name in ("query_diagnostics.csv", "pair_histograms.csv", "verification_ROC.csv", "verification_by_species.csv", "geometry_sample.csv")]
    if point_path.is_file() and all(p.is_file() for p in required):
        stored = json.loads(point_path.read_text())
        if stable_json(stored["provenance"]) == stable_json(provenance):
            geometry_ids = pd.read_csv(directory / "geometry_sample.csv", dtype={"image_id": str}).image_id
            combined = pd.concat([g, q], ignore_index=True).drop_duplicates("image_id")
            matrix = np.vstack([gz, qz])
            # Closed query/gallery are disjoint; pooled diagnostics use q==g.
            first_positions = pd.Series(np.arange(len(pd.concat([g, q], ignore_index=True))), index=pd.concat([g, q], ignore_index=True).image_id)
            first_positions = first_positions[~first_positions.index.duplicated()]
            positions = first_positions.loc[geometry_ids].to_numpy(int)
            enriched = pd.concat([g, q], ignore_index=True).iloc[positions].reset_index(drop=True)
            return stored["metrics"], pd.read_csv(required[1]), pd.read_csv(required[2]), enriched, matrix[positions], True
    result, hard, pairs, hist, curve, per_species = rep_pair_diagnostics(q, qz, g, gz, REP_CONFIG, rep_seed(REP_SEED, dataset, split_seed, "pairs"))
    combined = pd.concat([g, q], ignore_index=True)
    matrix = np.vstack([gz, qz])
    keep = ~combined.image_id.duplicated()
    combined, matrix = combined.loc[keep].reset_index(drop=True), matrix[keep]
    geometry, geometry_detail, chosen = rep_geometry(combined, matrix, REP_CONFIG, rep_seed(REP_SEED, dataset, "geometry"))
    result.update(geometry)
    result["species_metadata_fraction"] = float(combined.rep_species.ne("").mean())
    result["observation_group_metadata_fraction"] = float(combined.rep_observation_group.ne("").mean())
    result["species_metadata_status"] = "complete" if combined.rep_species.ne("").all() else ("partial" if combined.rep_species.ne("").any() else "missing; same-species metrics unavailable")
    result["pair_exclusion_policy"] = "exclude self; exclude equal nonempty observation groups; missing groups permit image-disjoint pairs"
    # Keep a compact plot curve. Metrics were computed from the complete ROC.
    if len(curve):
        curve = pd.concat([part.iloc[np.unique(np.linspace(0, len(part)-1, min(len(part), 1000)).astype(int))] for _, part in curve.groupby("pair_scope", sort=True)], ignore_index=True)
    atomic_write_csv(required[0], hard)
    atomic_write_csv(required[1], hist if len(hist) else pd.DataFrame(columns=["pair_group", "bin_left", "bin_right", "pair_count", "identity_balanced_probability_mass"]))
    atomic_write_csv(required[2], curve)
    atomic_write_csv(required[3], per_species if len(per_species) else pd.DataFrame(columns=["species", "AUROC", "status"]))
    atomic_write_csv(required[4], geometry_detail)
    if EXPORT_PAIR_SCORES:
        atomic_save_npz(directory / "sampled_pairs.npz", score=pairs.score.to_numpy(float), query_id=pairs.query_id.to_numpy(dtype="U"), gallery_id=pairs.gallery_id.to_numpy(dtype="U"), identity=pairs.identity.to_numpy(dtype="U"), species=pairs.species.to_numpy(dtype="U"), pair_group=pairs.pair_group.to_numpy(dtype="U"), provenance_json=np.asarray(stable_json(provenance)))
    write_json(point_path, {"provenance": provenance, "metrics": result})
    return result, hist, curve, combined.iloc[chosen].reset_index(drop=True), matrix[chosen], False


## Run closed-set retrieval and representation diagnostics

In [22]:
closed_rows, representation_rows, pool_rows, histogram_frames, roc_frames, status_rows = [], [], [], [], [], []
closed_detail_paths, membership_fingerprints = {}, {}
for model_name in MODEL_NAMES:
    pooled_samples = {seed: [] for seed in SPLIT_SEEDS}
    for dataset_name in EXTERNAL_DATASETS:
        frame, embeddings, embedding_metadata = canonical_embeddings_for(model_name, dataset_name)
        print(f"Evaluating {model_name} / {dataset_name}")
        for split_seed in SPLIT_SEEDS:
            start = time.perf_counter()
            split = split_manifests[(dataset_name, split_seed)]
            gallery_rows, gallery_embeddings = rows_for_roles(frame, embeddings, split, "closed_role", {"gallery"})
            query_rows, query_embeddings = rows_for_roles(frame, embeddings, split, "closed_role", {"query_known"})
            closed_meta = {"schema_version": 2, "benchmark_version": "closed-set-frozen-representation-v1", "model": model_name, "dataset": dataset_name, "split_seed": int(split_seed), "embedding_artifact_sha256": embedding_file_hashes[(model_name, dataset_name)], "split_manifest_sha256": split_manifest_hashes[(dataset_name, split_seed)], "query_image_ids": query_rows.image_id.tolist(), "gallery_image_ids": gallery_rows.image_id.tolist(), "closed_metric_implementation_sha256": CLOSED_SET_PRE_LOGISTIC_FUNCTION_SHA256}
            cache = PATHS["scores"] / model_name / dataset_name / f"seed_{split_seed}_closed_results.npz"
            cache_hit = False
            if cache.is_file():
                try:
                    metrics, details = load_closed_result_cache(cache, expected_metadata=closed_meta)
                    if not np.array_equal(details.image_id, query_rows.image_id) or not np.array_equal(details.identity, query_rows.identity):
                        raise ValueError("Cached query alignment differs")
                    cache_hit = True
                except (ValueError, KeyError, OSError):
                    pass
            if not cache_hit:
                metrics, details = closed_set_retrieval_metrics(query_embeddings=query_embeddings, query_identities=query_rows.identity, gallery_embeddings=gallery_embeddings, gallery_identities=gallery_rows.identity, chunk_size=SIMILARITY_CHUNK_SIZE)
                details.insert(0, "image_id", query_rows.image_id.to_numpy())
                save_closed_result_cache(cache, metadata=closed_meta, metrics=metrics, query_details=details)
            detail_path = PATHS["closed_set"] / model_name / dataset_name / f"seed_{split_seed}_query_details.csv"
            atomic_write_csv(detail_path, details)
            closed_detail_paths[(model_name, dataset_name, split_seed)] = detail_path
            bp = PATHS["statistics"] / "closed_set" / model_name / dataset_name / f"seed_{split_seed}.npz"
            bmeta = {"score_sha256": sha256_file(cache), "replicates": BOOTSTRAP_REPLICATES, "seed": BOOTSTRAP_SEED}
            verified_bootstrap(bp, expected_metadata=bmeta, expected_metrics=("Rank-1", "mAP"), expected_replicates=BOOTSTRAP_REPLICATES, compute=lambda: bootstrap_closed_details(details, replicates=BOOTSTRAP_REPLICATES, seed=stable_seed(BOOTSTRAP_SEED, "closed", dataset_name, split_seed)))
            closed_rows.append({"model": model_name, "dataset": dataset_name, "split_seed": split_seed, "n_gallery_samples": len(gallery_rows), "n_query_samples": len(query_rows), "n_gallery_identities": gallery_rows.identity.nunique(), **metrics, "bootstrap_path": str(bp)})
            rep, hist, roc, sampled_frame, sampled_z, rep_hit = run_representation_cached(model_name, dataset_name, split_seed, query_rows, query_embeddings, gallery_rows, gallery_embeddings, embedding_file_hashes[(model_name, dataset_name)])
            key = (dataset_name, split_seed)
            fingerprint = (rep["pair_membership_sha256"], rep["geometry_membership_sha256"])
            if key in membership_fingerprints and membership_fingerprints[key] != fingerprint:
                raise ValueError("Diagnostic samples differ between models; do not compare these results")
            membership_fingerprints[key] = fingerprint
            context = {"model": model_name, "dataset": dataset_name, "split_seed": split_seed}
            representation_rows.append({**context, **rep})
            histogram_frames.append(hist.assign(**context))
            roc_frames.append(roc.assign(**context))
            status_rows.append({**context, "species_status": rep["species_metadata_status"], "species_fraction": rep["species_metadata_fraction"], "group_fraction": rep["observation_group_metadata_fraction"], "same_species_pair_status": rep["verification_same_species_status"], "geometry_samples": rep["geometry_samples"], "silhouette_same_species_eligible_samples": rep["silhouette_same_species_eligible_samples"], "clustering_status": rep["closed_clustering_status"], "cache_hit": rep_hit})
            pooled_samples[split_seed].append((dataset_name, sampled_frame, sampled_z))
            runtime_recorder.record("closed_set_and_representation", time.perf_counter()-start, model=model_name, dataset=dataset_name, split_seed=split_seed, cache_hit=cache_hit and rep_hit)
    if RUN_POOLED_SPECIES_DIAGNOSTICS:
        for split_seed, pieces in pooled_samples.items():
            frames, matrices = [], []
            for dataset, f, z in pieces:
                f = f.copy()
                # IDs are dataset-scoped. This does not assert cross-dataset animal matches.
                f["image_id"] = [stable_json([dataset, value]) for value in f.image_id]
                f["identity"] = [stable_json([dataset, value]) for value in f.identity]
                frames.append(f)
                matrices.append(z)
            pool = pd.concat(frames, ignore_index=True)
            if len(set(pool.rep_species)-{""}) < 2:
                pool_rows.append({"model": model_name, "dataset": "pooled_species_diagnostic", "split_seed": split_seed, "status": "unavailable: fewer than two explicitly labeled species"})
                continue
            z = np.vstack(matrices)
            embedding_hash = sha256_bytes(stable_json({d: embedding_file_hashes[(model_name, d)] for d in EXTERNAL_DATASETS}).encode())
            rep, hist, roc, _, _, rep_hit = run_representation_cached(model_name, "pooled_species_diagnostic", split_seed, pool, z, pool, z, embedding_hash)
            key = ("pooled_species_diagnostic", split_seed)
            fingerprint = (rep["pair_membership_sha256"], rep["geometry_membership_sha256"])
            if key in membership_fingerprints and membership_fingerprints[key] != fingerprint:
                raise ValueError("Pooled diagnostic sampling differs between models")
            membership_fingerprints[key] = fingerprint
            context = {"model": model_name, "dataset": "pooled_species_diagnostic", "split_seed": split_seed}
            pool_rows.append({**context, **rep, "status": "diagnostic_only; species_and_dataset_may_be_confounded"})
            histogram_frames.append(hist.assign(**context))
            roc_frames.append(roc.assign(**context))
closed_set_results = pd.DataFrame(closed_rows)
representation_results = pd.DataFrame(representation_rows)
pooled_species_results = pd.DataFrame(pool_rows, columns=None if pool_rows else ["model", "dataset", "split_seed", "status"])
pair_histograms = pd.concat(histogram_frames, ignore_index=True)
verification_curves = pd.concat(roc_frames, ignore_index=True)
diagnostic_status = pd.DataFrame(status_rows)
for name, table in (("closed_set_results", closed_set_results), ("representation_results", representation_results), ("pooled_species_results", pooled_species_results), ("pair_similarity_histograms", pair_histograms), ("verification_ROC_curves", verification_curves), ("representation_status", diagnostic_status)):
    atomic_write_csv(PATHS["tables"] / f"{name}.csv", table)
display(closed_set_results.head())
display(diagnostic_status)


Evaluating ApexID_Convnext_Reweighted / bristol_gorillas
Evaluating ApexID_Convnext_Reweighted / elephants
Evaluating ApexID_Convnext_Reweighted / gcn_id
Evaluating ApexID_Convnext_Reweighted / jaguar_antonio


,model,dataset,split_seed,n_gallery_samples,n_query_samples,n_gallery_identities,Rank-1,Rank-5,Rank-10,CMC@20,CMC@50,mAP,identity-balanced Rank-1,effective_rank_5,effective_rank_10,effective_rank_20,effective_rank_50,bootstrap_path
0,ApexID_Convnext_Reweighted,bristol_gorillas,42,4215,1109,7,0.918846,0.976555,0.985573,0.996393,1.000000,0.259355,0.919022,5,10,20,50,/kaggle/working/apexid_cvpr_closed_representat...
1,ApexID_Convnext_Reweighted,bristol_gorillas,123,4234,1090,7,0.915596,0.967890,0.984404,0.990826,0.996330,0.267968,0.915446,5,10,20,50,/kaggle/working/apexid_cvpr_closed_representat...
2,ApexID_Convnext_Reweighted,bristol_gorillas,3407,4240,1084,7,0.909594,0.970480,0.985240,0.992620,0.996310,0.267742,0.913242,5,10,20,50,/kaggle/working/apexid_cvpr_closed_representat...
3,ApexID_Convnext_Reweighted,bristol_gorillas,2026,4218,1106,7,0.910488,0.972875,0.989150,0.992767,0.999096,0.264503,0.915369,5,10,20,50,/kaggle/working/apexid_cvpr_closed_representat...
4,ApexID_Convnext_Reweighted,bristol_gorillas,1337,4226,1098,7,0.933515,0.977231,0.987250,0.992714,0.999089,0.266111,0.934957,5,10,20,50,/kaggle/working/apexid_cvpr_closed_representat...


,model,dataset,split_seed,species_status,species_fraction,group_fraction,same_species_pair_status,geometry_samples,silhouette_same_species_eligible_samples,clustering_status,cache_hit
0,ApexID_Convnext_Reweighted,bristol_gorillas,42,complete,1.0,1.0,descriptive_test_ROC_not_a_deployment_threshold,84,84,conditional_known_K_balanced_sample,False
1,ApexID_Convnext_Reweighted,bristol_gorillas,123,complete,1.0,1.0,descriptive_test_ROC_not_a_deployment_threshold,84,84,conditional_known_K_balanced_sample,False
2,ApexID_Convnext_Reweighted,bristol_gorillas,3407,complete,1.0,1.0,descriptive_test_ROC_not_a_deployment_threshold,84,84,conditional_known_K_balanced_sample,False
3,ApexID_Convnext_Reweighted,bristol_gorillas,2026,complete,1.0,1.0,descriptive_test_ROC_not_a_deployment_threshold,84,84,conditional_known_K_balanced_sample,False
4,ApexID_Convnext_Reweighted,bristol_gorillas,1337,complete,1.0,1.0,descriptive_test_ROC_not_a_deployment_threshold,84,84,conditional_known_K_balanced_sample,False
5,ApexID_Convnext_Reweighted,elephants,42,missing; same-species metrics unavailable,0.0,1.0,no_eligible_positive_and_negative_pairs,1024,0,conditional_known_K_balanced_sample,False
6,ApexID_Convnext_Reweighted,elephants,123,missing; same-species metrics unavailable,0.0,1.0,no_eligible_positive_and_negative_pairs,1024,0,conditional_known_K_balanced_sample,False
7,ApexID_Convnext_Reweighted,elephants,3407,missing; same-species metrics unavailable,0.0,1.0,no_eligible_positive_and_negative_pairs,1024,0,conditional_known_K_balanced_sample,False
8,ApexID_Convnext_Reweighted,elephants,2026,missing; same-species metrics unavailable,0.0,1.0,no_eligible_positive_and_negative_pairs,1024,0,conditional_known_K_balanced_sample,False
9,ApexID_Convnext_Reweighted,elephants,1337,missing; same-species metrics unavailable,0.0,1.0,no_eligible_positive_and_negative_pairs,1024,0,conditional_known_K_balanced_sample,False


## Aggregate retrieval and representation results

In [23]:
closed_set_summary = hierarchical_bootstrap_summary(closed_set_results, metrics=("Rank-1", "mAP"), bootstrap_path_column="bootstrap_path", group_columns=("model", "dataset"), replicates=BOOTSTRAP_REPLICATES, seed=BOOTSTRAP_SEED)
additional_rows = []
for (model, dataset), group in closed_set_results.groupby(["model", "dataset"]):
    for metric in ("Rank-5", "Rank-10", "CMC@20", "CMC@50", "identity-balanced Rank-1"):
        additional_rows.append({"model": model, "dataset": dataset, "metric": metric, "mean": group[metric].mean(), "ci_low": np.nan, "ci_high": np.nan, "n_splits": len(group), "bootstrap_replicates": 0, "bootstrap_method": "point estimate; CI not computed"})
closed_set_summary = pd.concat([closed_set_summary, pd.DataFrame(additional_rows)], ignore_index=True)
dataset_macro_summary = dataset_macro_hierarchical_bootstrap(closed_set_results, task="closed_set", metrics=("Rank-1", "mAP"), bootstrap_path_column="bootstrap_path", replicates=BOOTSTRAP_REPLICATES, seed=BOOTSTRAP_SEED)
metric_registry = rep_metric_registry(REP_CONFIG["far_targets"])
representation_summary, representation_macro_summary = rep_summaries(representation_results, metric_registry, len(SPLIT_SEEDS), EXTERNAL_DATASETS)
for name, table in (("closed_set_summary", closed_set_summary), ("dataset_macro_summary", dataset_macro_summary), ("representation_summary", representation_summary), ("representation_macro_summary", representation_macro_summary), ("representation_metric_registry", pd.DataFrame([{"metric": k, "direction": v[0], "unit": v[1]} for k, v in metric_registry.items()]))):
    atomic_write_csv(PATHS["tables"] / f"{name}.csv", table)
display(dataset_macro_summary)
display(representation_macro_summary.pivot(index="model", columns="metric", values="mean"))


,task,model,metric,aggregation,mean,ci_low,ci_high,number_datasets,bootstrap_replicates,bootstrap_method
0,closed_set,ApexID_Convnext_Reweighted,Rank-1,Dataset-Macro Average,0.640549,0.338628,0.894035,4,1000,dataset + split-seed + identity-block hierarch...
1,closed_set,ApexID_Convnext_Reweighted,mAP,Dataset-Macro Average,0.305193,0.113747,0.530220,4,1000,dataset + split-seed + identity-block hierarch...


metric,alignment_squared_l2,closed_clustering_ARI_known_K,closed_clustering_B3_F1_known_K,closed_clustering_B3_precision_known_K,closed_clustering_B3_recall_known_K,cosine_mean_negative_all,cosine_mean_negative_cross_species,cosine_mean_negative_same_species,cosine_mean_positive,effective_rank_centered,...,silhouette_species,species_gap,uniformity_log_mean_exp,verification_all_AUROC,verification_all_TAR_at_FAR_0.01,verification_all_achieved_FAR_0.01,verification_same_species_AUROC,verification_same_species_TAR_at_FAR_0.01,verification_same_species_achieved_FAR_0.01,verification_same_species_macro_AUROC
model,,,,,,,,,,,,,,,,,,,,,
ApexID_Convnext_Reweighted,0.957911,0.110919,0.421298,0.35923,0.515526,0.396429,NaN,NaN,0.521045,282.845215,...,NaN,NaN,-2.299965,0.720001,0.228643,0.009828,NaN,NaN,NaN,NaN


## Compare every model pair

In [24]:
from itertools import combinations
paired_frames, diagnostic_pair_rows = [], []
if RUN_PAIRED_COMPARISONS:
    for reference, comparison in combinations(MODEL_NAMES, 2):
        for dataset in EXTERNAL_DATASETS:
            paths = []
            for split_seed in SPLIT_SEEDS:
                refpath = closed_detail_paths[(reference, dataset, split_seed)]
                cmppath = closed_detail_paths[(comparison, dataset, split_seed)]
                ref = pd.read_csv(refpath, dtype={"image_id": str, "identity": str})
                cmp = pd.read_csv(cmppath, dtype={"image_id": str, "identity": str})
                path = PATHS["statistics"] / "paired_closed" / reference / comparison / dataset / f"seed_{split_seed}.npz"
                metadata = {"reference_details_sha256": sha256_file(refpath), "comparison_details_sha256": sha256_file(cmppath), "replicates": BOOTSTRAP_REPLICATES, "seed": BOOTSTRAP_SEED}
                verified_bootstrap(path, expected_metadata=metadata, expected_metrics=("Rank-1", "mAP"), expected_replicates=BOOTSTRAP_REPLICATES, compute=lambda: paired_closed_bootstrap_difference(ref, cmp, replicates=BOOTSTRAP_REPLICATES, seed=stable_seed(BOOTSTRAP_SEED, reference, comparison, dataset, split_seed)))
                paths.append(path)
            a = closed_set_results.loc[(closed_set_results.model == reference) & (closed_set_results.dataset == dataset)]
            b = closed_set_results.loc[(closed_set_results.model == comparison) & (closed_set_results.dataset == dataset)]
            paired_frames.append(paired_difference_summary(bootstrap_paths=paths, point_differences={m: float(a[m].mean()-b[m].mean()) for m in ("Rank-1", "mAP")}, metrics=("Rank-1", "mAP"), replicates=BOOTSTRAP_REPLICATES, seed=stable_seed(BOOTSTRAP_SEED, reference, comparison, dataset), metadata={"reference_model": reference, "comparison_model": comparison, "dataset": dataset, "task": "closed_set"}))
            a = representation_results.loc[(representation_results.model == reference) & (representation_results.dataset == dataset)].set_index("split_seed")
            b = representation_results.loc[(representation_results.model == comparison) & (representation_results.dataset == dataset)].set_index("split_seed")
            for metric, (direction, unit) in metric_registry.items():
                difference = pd.to_numeric(a[metric], errors="coerce")-pd.to_numeric(b[metric], errors="coerce")
                valid = difference[np.isfinite(difference)]
                diagnostic_pair_rows.append({"reference_model": reference, "comparison_model": comparison, "dataset": dataset, "metric": metric, "mean_difference_reference_minus_comparison": valid.mean(), "split_difference_sd": valid.std(ddof=1), "positive_differences": int((valid > 1e-12).sum()), "ties": int((valid.abs() <= 1e-12).sum()), "negative_differences": int((valid < -1e-12).sum()), "n_valid_splits": len(valid), "direction": direction, "unit": unit, "uncertainty": "descriptive; overlapping splits are not independent trials"})
paired_model_comparisons = pd.concat(paired_frames, ignore_index=True) if paired_frames else pd.DataFrame(columns=["reference_model", "comparison_model", "dataset", "metric"])
paired_representation_comparisons = pd.DataFrame(diagnostic_pair_rows, columns=None if diagnostic_pair_rows else ["reference_model", "comparison_model", "dataset", "metric"])
atomic_write_csv(PATHS["tables"] / "paired_model_comparisons.csv", paired_model_comparisons)
atomic_write_csv(PATHS["tables"] / "paired_representation_comparisons.csv", paired_representation_comparisons)
display(paired_representation_comparisons.head(15))


,reference_model,comparison_model,dataset,metric


## Generate closed-set figures

In [25]:
plot_paths = []
if RUN_PLOTS:
    import matplotlib.pyplot as plt
    def save_figure(fig, name):
        for suffix in ("png", "pdf"):
            path = PATHS["plots"] / f"{name}.{suffix}"
            fig.savefig(path, dpi=180, bbox_inches="tight")
            plot_paths.append(str(path))
        plt.close(fig)
    for dataset in EXTERNAL_DATASETS:
        fig, axes = plt.subplots(1, 2, figsize=(11, 4))
        for ax, metric in zip(axes, ("Rank-1", "mAP")):
            rows = closed_set_summary.loc[(closed_set_summary.dataset == dataset) & (closed_set_summary.metric == metric)].set_index("model").reindex(MODEL_NAMES)
            ax.bar(np.arange(len(rows)), rows["mean"]*100)
            ax.set_xticks(np.arange(len(rows)), MODEL_NAMES, rotation=30, ha="right")
            ax.set_ylabel("Percent")
            ax.set_title(f"{dataset}: {metric}")
        fig.tight_layout()
        save_figure(fig, f"closed_retrieval_{dataset}")
    for dataset in pair_histograms.dataset.unique():
        fig, axes = plt.subplots(1, 3, figsize=(14, 4))
        for ax, pair_group in zip(axes, ("positive", "negative_same_species", "negative_cross_species")):
            for model in MODEL_NAMES:
                selected = pair_histograms.loc[(pair_histograms.dataset == dataset) & (pair_histograms.model == model) & (pair_histograms.pair_group == pair_group)]
                if selected.empty:
                    continue
                mean = selected.groupby(["bin_left", "bin_right"]).identity_balanced_probability_mass.mean().reset_index()
                ax.plot((mean.bin_left+mean.bin_right)/2, mean.identity_balanced_probability_mass, label=model)
            ax.set_title(pair_group.replace("_", " "))
            ax.set_xlabel("Cosine similarity")
            ax.set_ylabel("Identity-balanced probability mass")
            ax.set_xlim(-1, 1)
            if ax.lines:
                ax.legend(fontsize=7)
            else:
                ax.text(0.5, 0.5, "Unavailable: required species/pair coverage", transform=ax.transAxes, ha="center", fontsize=8)
        fig.suptitle(dataset)
        fig.tight_layout()
        save_figure(fig, f"similarity_distributions_{dataset}")
    for dataset in EXTERNAL_DATASETS:
        fig, axes = plt.subplots(1, 2, figsize=(11, 4))
        for ax, scope in zip(axes, ("all", "same_species")):
            for model in MODEL_NAMES:
                curves = verification_curves.loc[(verification_curves.dataset == dataset) & (verification_curves.model == model) & (verification_curves.pair_scope == scope)]
                for j, (_, curve) in enumerate(curves.groupby("split_seed")):
                    ax.plot(curve.FAR, curve.TAR, alpha=0.55, linewidth=1, label=model if j == 0 else None, color=f"C{MODEL_NAMES.index(model)%10}")
            ax.plot([0, 1], [0, 1], "k--", alpha=0.3)
            ax.set(xlabel="False acceptance rate", ylabel="True acceptance rate", title=f"{dataset}: {scope}", xlim=(0,1), ylim=(0,1))
            handles, labels = ax.get_legend_handles_labels()
            if labels:
                ax.legend(fontsize=7)
        fig.tight_layout()
        save_figure(fig, f"verification_ROC_{dataset}")
    focus = ["verification_same_species_AUROC", "identity_gap_same_species", "hard_margin_same_species_mean", "silhouette_identity_same_species", "effective_rank_uncentered_fraction", "closed_clustering_B3_F1_known_K"]
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    for ax, metric in zip(axes.ravel(), focus):
        for j, model in enumerate(MODEL_NAMES):
            part = representation_summary.loc[(representation_summary.model == model) & (representation_summary.metric == metric)].set_index("dataset").reindex(EXTERNAL_DATASETS)
            ax.plot(np.arange(len(EXTERNAL_DATASETS)), part["mean"], marker="o", label=model)
        ax.set_title(metric.replace("_", " "), fontsize=9)
        if metric_registry[metric][1] == "0_to_1":
            ax.set_ylim(0, 1.02)
        ax.set_xticks(np.arange(len(EXTERNAL_DATASETS)), EXTERNAL_DATASETS, rotation=25, ha="right", fontsize=8)
        ax.grid(alpha=0.2)
    axes[0,0].legend(fontsize=7)
    fig.suptitle("Closed-set representation diagnostics: split means; no inferential error bars")
    fig.tight_layout()
    save_figure(fig, "representation_diagnostics")
print(f"Saved {len(plot_paths)} plot files.")


Saved 28 plot files.


## Validate and write the report

In [26]:
expected_rows = len(MODEL_NAMES)*len(EXTERNAL_DATASETS)*len(SPLIT_SEEDS)
if len(closed_set_results) != expected_rows or len(representation_results) != expected_rows:
    raise RuntimeError("Incomplete benchmark output")
if closed_set_results.duplicated(["model", "dataset", "split_seed"]).any():
    raise RuntimeError("Duplicate closed-set results")
if not np.isfinite(closed_set_results[["Rank-1", "mAP", "identity-balanced Rank-1"]]).all().all():
    raise RuntimeError("Nonfinite headline retrieval metrics")
runtime_recorder.record("benchmark_wall_clock", time.perf_counter()-BENCHMARK_WALL_START)
runtime_recorder.save(OUTPUT_ROOT / "runtime_report.csv")
manifest = {"version": STAGE02_VERSION, "status": "completed", "created_at_utc": datetime.now(timezone.utc).isoformat(), "models": model_metadata_by_name, "models_evaluated": list(MODEL_NAMES), "datasets": list(EXTERNAL_DATASETS), "split_seeds": list(SPLIT_SEEDS), "execution_guarantees": {"model_training": False, "embedding_extraction": False, "test_split_regeneration": False, "deployment_threshold_fitting": False}, "retrieval_metric_source_sha256": CLOSED_SET_PRE_LOGISTIC_FUNCTION_SHA256, "representation_implementation_sha256": REP_IMPLEMENTATION_SHA256, "representation_config": REP_CONFIG, "representation_seed": REP_SEED, "species_columns": SPECIES_COLUMNS, "observation_group_columns": OBSERVATION_GROUP_COLUMNS, "verified_dataset_species": VERIFIED_DATASET_SPECIES, "pooled_species_diagnostic": RUN_POOLED_SPECIES_DIAGNOSTICS, "bootstrap": {"replicates": BOOTSTRAP_REPLICATES, "seed": BOOTSTRAP_SEED, "scope": "existing retrieval Rank-1/mAP only; conditional on frozen embeddings; not retraining uncertainty"}, "new_diagnostic_uncertainty": "descriptive split SD only", "TAR_at_FAR_semantics": "empirical weighted test ROC; no interpolation, no deployment calibration", "closed_clustering_semantics": "cosine average-linkage; known K from represented closed identities; label-balanced sample; conditional diagnostic", "source_manifests": {"stage00_sha256": sha256_file(STAGE00_MANIFEST_PATH), "stage01_sha256": sha256_file(STAGE01_MANIFEST_PATH)}, "observed_source_file_hashes": observed_source_hashes, "embedding_artifact_hashes": {f"{m}/{d}": h for (m,d), h in embedding_file_hashes.items()}, "environment": environment_provenance(), "plot_files": plot_paths}
write_json(OUTPUT_ROOT / "benchmark_manifest.json", manifest)
write_json(OUTPUT_ROOT / "stage02_manifest.json", manifest)
primary_diagnostics = representation_macro_summary.pivot(index="metric", columns="model", values="mean").reindex(list(metric_registry)).reindex(columns=MODEL_NAMES).reset_index()
def report_table(frame):
    if not len(frame):
        return "No eligible rows."
    def value(v):
        if isinstance(v, (float, np.floating)):
            return f"{v:.5f}" if np.isfinite(v) else "unavailable"
        return str(v).replace("|", "/").replace("\n", " ")
    return "\n".join(["| "+" | ".join(map(str, frame.columns))+" |", "| "+" | ".join(["---"]*len(frame.columns))+" |", *["| "+" | ".join(map(value, row))+" |" for row in frame.itertuples(index=False, name=None)]])
report_markdown = "\n\n".join([
    "# ApexID closed-set benchmark and representation report",
    "## Evaluation scope\nFrozen gallery/query splits and cached embeddings only. The original image-gallery cosine Rank-1/mAP implementation is preserved. New diagnostics explain representation behavior and do not replace retrieval results.",
    "## Closed-set dataset macro\n"+report_table(dataset_macro_summary),
    "## Representation dataset macro\n"+report_table(primary_diagnostics),
    "## Metadata and eligibility\n"+report_table(diagnostic_status),
    "## Pooled species geometry\n"+report_table(pooled_species_results[[c for c in ("model", "split_seed", "cosine_mean_positive", "cosine_mean_negative_same_species", "cosine_mean_negative_cross_species", "identity_gap_same_species", "species_gap", "silhouette_species", "status") if c in pooled_species_results]]),
    "Pooled results are diagnostics only and excluded from dataset-macro averages. Different datasets can have different backgrounds/acquisition pipelines, so cross-dataset species separation is confounded with dataset cues. Dataset-scoped identities are not assumed to match across datasets.",
    "## Interpretation\nHigher pair AUROC/TAR, identity gaps, hard-negative margins and identity silhouette are favorable under the same sampling protocol. Lower hard-margin violation rates are favorable. Effective rank, species separation and uniformity are diagnostics, not universal quality scores. Alignment is 2 minus 2 times positive cosine similarity and is not independent evidence of quality. Geometry is measured on a bounded identity-balanced sample; compare rank only with its sample/dimension ceiling and comparable membership.",
    "## Statistical limits\nExisting retrieval intervals reuse the supplied split/identity-bootstrap approach and are conditional on frozen checkpoints. New diagnostic SDs describe overlapping evaluation splits; they are not independent training-run uncertainty or confidence intervals. Pair samples are not independent identity observations. Missing metrics remain NaN with coverage/status fields, and the headline diagnostic macro requires complete dataset coverage.",
    "## Verification protocol\nPositive pairs share identity; negatives have different identities. Same-species and cross-species negatives require explicit species metadata. Self-pairs and equal nonempty observation-group pairs are excluded. If group metadata are missing, only image disjointness is guaranteed. Pairs are selected using metadata and common seeds, with equal identity/query weighting and bounded sampling. TAR at FAR is a conservative empirical test-ROC point with achieved FAR and threshold recorded for diagnosis only; no threshold is transferred into prediction.",
    "## Closed-set clustering\nBÂ³ precision/recall/F1 and ARI use cosine average-linkage on the geometry sample, cut to the known number K of represented closed-set identities. This uses class-count information and balanced sampling; it is a conditional diagnostic, not unknown-identity discovery. Ground-truth labels are used for metric calculation, and no clustering threshold is selected on test performance.",
    "## Files\nSee tables/ for results, metric directions/units, coverage, descriptive all-pair model comparisons, score histograms and ROC curves. representation/ contains per-query margins, geometry sample membership/silhouettes and provenance-checked diagnostic caches. plots/ includes PNG and PDF figures. Species-dependent metrics need real species labels or verified dataset overrides.",
]) + "\n"
atomic_write_text(OUTPUT_ROOT / "final_benchmark_report.md", report_markdown)
write_json(OUTPUT_ROOT / "final_benchmark_report.json", {"manifest": manifest, "closed_macro": dataset_macro_summary.to_dict("records"), "representation_macro": representation_macro_summary.to_dict("records"), "status": diagnostic_status.to_dict("records")})
atomic_write_text(OUTPUT_ROOT / "paper_tables.md", "# Closed-set tables\n\n"+report_table(dataset_macro_summary)+"\n\n"+report_table(primary_diagnostics)+"\n")
display(Markdown(report_markdown))


# ApexID closed-set benchmark and representation report

## Evaluation scope
Frozen gallery/query splits and cached embeddings only. The original image-gallery cosine Rank-1/mAP implementation is preserved. New diagnostics explain representation behavior and do not replace retrieval results.

## Closed-set dataset macro
| task | model | metric | aggregation | mean | ci_low | ci_high | number_datasets | bootstrap_replicates | bootstrap_method |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| closed_set | ApexID_Convnext_Reweighted | Rank-1 | Dataset-Macro Average | 0.64055 | 0.33863 | 0.89403 | 4 | 1000 | dataset + split-seed + identity-block hierarchical bootstrap |
| closed_set | ApexID_Convnext_Reweighted | mAP | Dataset-Macro Average | 0.30519 | 0.11375 | 0.53022 | 4 | 1000 | dataset + split-seed + identity-block hierarchical bootstrap |

## Representation dataset macro
| metric | ApexID_Convnext_Reweighted |
| --- | --- |
| verification_all_AUROC | 0.72000 |
| verification_same_species_AUROC | unavailable |
| verification_same_species_macro_AUROC | unavailable |
| cosine_mean_positive | 0.52104 |
| cosine_mean_negative_all | 0.39643 |
| cosine_mean_negative_same_species | unavailable |
| cosine_mean_negative_cross_species | unavailable |
| identity_gap_all | 0.12462 |
| identity_gap_same_species | unavailable |
| species_gap | unavailable |
| alignment_squared_l2 | 0.95791 |
| silhouette_identity_all | 0.00220 |
| silhouette_identity_same_species | unavailable |
| silhouette_species | unavailable |
| effective_rank_uncentered | 252.08037 |
| effective_rank_centered | 282.84522 |
| effective_rank_uncentered_fraction | 0.52669 |
| effective_rank_centered_fraction | 0.57478 |
| uniformity_log_mean_exp | -2.29996 |
| closed_clustering_B3_precision_known_K | 0.35923 |
| closed_clustering_B3_recall_known_K | 0.51553 |
| closed_clustering_B3_F1_known_K | 0.42130 |
| closed_clustering_ARI_known_K | 0.11092 |
| hard_margin_all_mean | -0.16702 |
| hard_margin_all_median | -0.16597 |
| hard_margin_all_p10 | -0.28537 |
| hard_margin_all_violation_rate | 0.84078 |
| hard_margin_same_species_mean | unavailable |
| hard_margin_same_species_median | unavailable |
| hard_margin_same_species_p10 | unavailable |
| hard_margin_same_species_violation_rate | unavailable |
| verification_all_TAR_at_FAR_0.01 | 0.22864 |
| verification_all_achieved_FAR_0.01 | 0.00983 |
| verification_same_species_TAR_at_FAR_0.01 | unavailable |
| verification_same_species_achieved_FAR_0.01 | unavailable |

## Metadata and eligibility
| model | dataset | split_seed | species_status | species_fraction | group_fraction | same_species_pair_status | geometry_samples | silhouette_same_species_eligible_samples | clustering_status | cache_hit |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| ApexID_Convnext_Reweighted | bristol_gorillas | 42 | complete | 1.00000 | 1.00000 | descriptive_test_ROC_not_a_deployment_threshold | 84 | 84 | conditional_known_K_balanced_sample | False |
| ApexID_Convnext_Reweighted | bristol_gorillas | 123 | complete | 1.00000 | 1.00000 | descriptive_test_ROC_not_a_deployment_threshold | 84 | 84 | conditional_known_K_balanced_sample | False |
| ApexID_Convnext_Reweighted | bristol_gorillas | 3407 | complete | 1.00000 | 1.00000 | descriptive_test_ROC_not_a_deployment_threshold | 84 | 84 | conditional_known_K_balanced_sample | False |
| ApexID_Convnext_Reweighted | bristol_gorillas | 2026 | complete | 1.00000 | 1.00000 | descriptive_test_ROC_not_a_deployment_threshold | 84 | 84 | conditional_known_K_balanced_sample | False |
| ApexID_Convnext_Reweighted | bristol_gorillas | 1337 | complete | 1.00000 | 1.00000 | descriptive_test_ROC_not_a_deployment_threshold | 84 | 84 | conditional_known_K_balanced_sample | False |
| ApexID_Convnext_Reweighted | elephants | 42 | missing; same-species metrics unavailable | 0.00000 | 1.00000 | no_eligible_positive_and_negative_pairs | 1024 | 0 | conditional_known_K_balanced_sample | False |
| ApexID_Convnext_Reweighted | elephants | 123 | missing; same-species metrics unavailable | 0.00000 | 1.00000 | no_eligible_positive_and_negative_pairs | 1024 | 0 | conditional_known_K_balanced_sample | False |
| ApexID_Convnext_Reweighted | elephants | 3407 | missing; same-species metrics unavailable | 0.00000 | 1.00000 | no_eligible_positive_and_negative_pairs | 1024 | 0 | conditional_known_K_balanced_sample | False |
| ApexID_Convnext_Reweighted | elephants | 2026 | missing; same-species metrics unavailable | 0.00000 | 1.00000 | no_eligible_positive_and_negative_pairs | 1024 | 0 | conditional_known_K_balanced_sample | False |
| ApexID_Convnext_Reweighted | elephants | 1337 | missing; same-species metrics unavailable | 0.00000 | 1.00000 | no_eligible_positive_and_negative_pairs | 1024 | 0 | conditional_known_K_balanced_sample | False |
| ApexID_Convnext_Reweighted | gcn_id | 42 | missing; same-species metrics unavailable | 0.00000 | 1.00000 | no_eligible_positive_and_negative_pairs | 1024 | 0 | conditional_known_K_balanced_sample | False |
| ApexID_Convnext_Reweighted | gcn_id | 123 | missing; same-species metrics unavailable | 0.00000 | 1.00000 | no_eligible_positive_and_negative_pairs | 1024 | 0 | conditional_known_K_balanced_sample | False |
| ApexID_Convnext_Reweighted | gcn_id | 3407 | missing; same-species metrics unavailable | 0.00000 | 1.00000 | no_eligible_positive_and_negative_pairs | 1024 | 0 | conditional_known_K_balanced_sample | False |
| ApexID_Convnext_Reweighted | gcn_id | 2026 | missing; same-species metrics unavailable | 0.00000 | 1.00000 | no_eligible_positive_and_negative_pairs | 1024 | 0 | conditional_known_K_balanced_sample | False |
| ApexID_Convnext_Reweighted | gcn_id | 1337 | missing; same-species metrics unavailable | 0.00000 | 1.00000 | no_eligible_positive_and_negative_pairs | 1024 | 0 | conditional_known_K_balanced_sample | False |
| ApexID_Convnext_Reweighted | jaguar_antonio | 42 | complete | 1.00000 | 1.00000 | descriptive_test_ROC_not_a_deployment_threshold | 405 | 405 | conditional_known_K_balanced_sample | False |
| ApexID_Convnext_Reweighted | jaguar_antonio | 123 | complete | 1.00000 | 1.00000 | descriptive_test_ROC_not_a_deployment_threshold | 405 | 405 | conditional_known_K_balanced_sample | False |
| ApexID_Convnext_Reweighted | jaguar_antonio | 3407 | complete | 1.00000 | 1.00000 | descriptive_test_ROC_not_a_deployment_threshold | 405 | 405 | conditional_known_K_balanced_sample | False |
| ApexID_Convnext_Reweighted | jaguar_antonio | 2026 | complete | 1.00000 | 1.00000 | descriptive_test_ROC_not_a_deployment_threshold | 405 | 405 | conditional_known_K_balanced_sample | False |
| ApexID_Convnext_Reweighted | jaguar_antonio | 1337 | complete | 1.00000 | 1.00000 | descriptive_test_ROC_not_a_deployment_threshold | 405 | 405 | conditional_known_K_balanced_sample | False |

## Pooled species geometry
| model | split_seed | cosine_mean_positive | cosine_mean_negative_same_species | cosine_mean_negative_cross_species | identity_gap_same_species | species_gap | silhouette_species | status |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| ApexID_Convnext_Reweighted | 42 | 0.64943 | 0.18120 | 0.02311 | 0.15523 | 0.15809 | 0.19592 | diagnostic_only; species_and_dataset_may_be_confounded |
| ApexID_Convnext_Reweighted | 123 | 0.64887 | 0.17808 | 0.02292 | 0.14992 | 0.15516 | 0.19592 | diagnostic_only; species_and_dataset_may_be_confounded |
| ApexID_Convnext_Reweighted | 3407 | 0.65004 | 0.18273 | 0.02163 | 0.15851 | 0.16110 | 0.19592 | diagnostic_only; species_and_dataset_may_be_confounded |
| ApexID_Convnext_Reweighted | 2026 | 0.64852 | 0.17535 | 0.02017 | 0.15193 | 0.15518 | 0.19592 | diagnostic_only; species_and_dataset_may_be_confounded |
| ApexID_Convnext_Reweighted | 1337 | 0.64854 | 0.17446 | 0.02303 | 0.15185 | 0.15143 | 0.19592 | diagnostic_only; species_and_dataset_may_be_confounded |

Pooled results are diagnostics only and excluded from dataset-macro averages. Different datasets can have different backgrounds/acquisition pipelines, so cross-dataset species separation is confounded with dataset cues. Dataset-scoped identities are not assumed to match across datasets.

## Interpretation
Higher pair AUROC/TAR, identity gaps, hard-negative margins and identity silhouette are favorable under the same sampling protocol. Lower hard-margin violation rates are favorable. Effective rank, species separation and uniformity are diagnostics, not universal quality scores. Alignment is 2 minus 2 times positive cosine similarity and is not independent evidence of quality. Geometry is measured on a bounded identity-balanced sample; compare rank only with its sample/dimension ceiling and comparable membership.

## Statistical limits
Existing retrieval intervals reuse the supplied split/identity-bootstrap approach and are conditional on frozen checkpoints. New diagnostic SDs describe overlapping evaluation splits; they are not independent training-run uncertainty or confidence intervals. Pair samples are not independent identity observations. Missing metrics remain NaN with coverage/status fields, and the headline diagnostic macro requires complete dataset coverage.

## Verification protocol
Positive pairs share identity; negatives have different identities. Same-species and cross-species negatives require explicit species metadata. Self-pairs and equal nonempty observation-group pairs are excluded. If group metadata are missing, only image disjointness is guaranteed. Pairs are selected using metadata and common seeds, with equal identity/query weighting and bounded sampling. TAR at FAR is a conservative empirical test-ROC point with achieved FAR and threshold recorded for diagnosis only; no threshold is transferred into prediction.

## Closed-set clustering
BÂ³ precision/recall/F1 and ARI use cosine average-linkage on the geometry sample, cut to the known number K of represented closed-set identities. This uses class-count information and balanced sampling; it is a conditional diagnostic, not unknown-identity discovery. Ground-truth labels are used for metric calculation, and no clustering threshold is selected on test performance.

## Files
See tables/ for results, metric directions/units, coverage, descriptive all-pair model comparisons, score histograms and ROC curves. representation/ contains per-query margins, geometry sample membership/silhouettes and provenance-checked diagnostic caches. plots/ includes PNG and PDF figures. Species-dependent metrics need real species labels or verified dataset overrides.


In [27]:
# Organized headline-number report


def _format_report_number(value):
    if pd.isna(value):
        return "-"
    if isinstance(value, (float, np.floating)):
        return f"{value:.4f}"
    return str(value).replace("|", "/").replace("\n", " ")


def _organized_table(frame, columns=None, rename=None):
    table = frame.copy()
    if columns is not None:
        table = table[[column for column in columns if column in table.columns]]
    if rename:
        table = table.rename(columns=rename)
    headers = list(table.columns)
    rows = [
        "| " + " | ".join(map(str, headers)) + " |",
        "| " + " | ".join(["---"] * len(headers)) + " |",
    ]
    rows.extend(
        "| " + " | ".join(_format_report_number(value) for value in row) + " |"
        for row in table.itertuples(index=False, name=None)
    )
    return "\n".join(rows)


retrieval_report = dataset_macro_summary.copy()
retrieval_report["95% CI"] = retrieval_report.apply(
    lambda row: "-" if pd.isna(row["ci_low"]) else f"[{row['ci_low']:.4f}, {row['ci_high']:.4f}]",
    axis=1,
)
retrieval_report = retrieval_report.sort_values(["model", "metric"]).reset_index(drop=True)
retrieval_report = retrieval_report.rename(columns={"model": "Model", "dataset": "Dataset", "metric": "Metric", "mean": "Mean"})
representation_report = primary_diagnostics.reset_index().rename(columns={"metric": "Metric"})
representation_report = representation_report.sort_values("Metric").reset_index(drop=True)

organized_sections = [
    "# ApexID closed-set benchmark and representation report",
    "## Headline retrieval results\nMean values use the native 0-1 scale; intervals are shown where bootstrap estimates are available.\n\n" + _organized_table(retrieval_report, ["Model", "Dataset", "Metric", "Mean", "95% CI"]),
    "## Representation diagnostics\nEach row is a diagnostic metric and each model column contains its dataset-macro value.\n\n" + _organized_table(representation_report),
    "## Eligibility and metadata\n" + _organized_table(diagnostic_status),
    "## Interpretation\nHigher Rank-1/mAP, pair AUROC/TAR, identity gaps, hard-negative margins and identity silhouette are favorable. Lower hard-margin violation rates are favorable. Representation diagnostics do not replace the closed-set retrieval results.",
    "## Statistical limits\nRetrieval intervals are conditional on frozen checkpoints. New diagnostic standard deviations describe overlapping evaluation splits and are not independent training-run confidence intervals.",
    "## Files\nSee `tables/` for the complete per-model/per-dataset results, metric units, coverage, comparisons, histograms and ROC data.",
]
organized_report = "\n\n".join(organized_sections) + "\n"
atomic_write_text(OUTPUT_ROOT / "final_benchmark_report.md", organized_report)
atomic_write_text(OUTPUT_ROOT / "paper_tables.md", "# Closed-set tables\n\n" + _organized_table(retrieval_report, ["Model", "Dataset", "Metric", "Mean", "95% CI"]) + "\n\n# Representation diagnostics\n\n" + _organized_table(representation_report) + "\n")
display(Markdown(organized_report))


# ApexID closed-set benchmark and representation report

## Headline retrieval results
Mean values use the native 0-1 scale; intervals are shown where bootstrap estimates are available.

| Model | Metric | Mean | 95% CI |
| --- | --- | --- | --- |
| ApexID_Convnext_Reweighted | Rank-1 | 0.6405 | [0.3386, 0.8940] |
| ApexID_Convnext_Reweighted | mAP | 0.3052 | [0.1137, 0.5302] |

## Representation diagnostics
Each row is a diagnostic metric and each model column contains its dataset-macro value.

| index | Metric | ApexID_Convnext_Reweighted |
| --- | --- | --- |
| 10 | alignment_squared_l2 | 0.9579 |
| 22 | closed_clustering_ARI_known_K | 0.1109 |
| 21 | closed_clustering_B3_F1_known_K | 0.4213 |
| 19 | closed_clustering_B3_precision_known_K | 0.3592 |
| 20 | closed_clustering_B3_recall_known_K | 0.5155 |
| 4 | cosine_mean_negative_all | 0.3964 |
| 6 | cosine_mean_negative_cross_species | - |
| 5 | cosine_mean_negative_same_species | - |
| 3 | cosine_mean_positive | 0.5210 |
| 15 | effective_rank_centered | 282.8452 |
| 17 | effective_rank_centered_fraction | 0.5748 |
| 14 | effective_rank_uncentered | 252.0804 |
| 16 | effective_rank_uncentered_fraction | 0.5267 |
| 23 | hard_margin_all_mean | -0.1670 |
| 24 | hard_margin_all_median | -0.1660 |
| 25 | hard_margin_all_p10 | -0.2854 |
| 26 | hard_margin_all_violation_rate | 0.8408 |
| 27 | hard_margin_same_species_mean | - |
| 28 | hard_margin_same_species_median | - |
| 29 | hard_margin_same_species_p10 | - |
| 30 | hard_margin_same_species_violation_rate | - |
| 7 | identity_gap_all | 0.1246 |
| 8 | identity_gap_same_species | - |
| 11 | silhouette_identity_all | 0.0022 |
| 12 | silhouette_identity_same_species | - |
| 13 | silhouette_species | - |
| 9 | species_gap | - |
| 18 | uniformity_log_mean_exp | -2.3000 |
| 0 | verification_all_AUROC | 0.7200 |
| 31 | verification_all_TAR_at_FAR_0.01 | 0.2286 |
| 32 | verification_all_achieved_FAR_0.01 | 0.0098 |
| 1 | verification_same_species_AUROC | - |
| 33 | verification_same_species_TAR_at_FAR_0.01 | - |
| 34 | verification_same_species_achieved_FAR_0.01 | - |
| 2 | verification_same_species_macro_AUROC | - |

## Eligibility and metadata
| model | dataset | split_seed | species_status | species_fraction | group_fraction | same_species_pair_status | geometry_samples | silhouette_same_species_eligible_samples | clustering_status | cache_hit |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| ApexID_Convnext_Reweighted | bristol_gorillas | 42 | complete | 1.0000 | 1.0000 | descriptive_test_ROC_not_a_deployment_threshold | 84 | 84 | conditional_known_K_balanced_sample | False |
| ApexID_Convnext_Reweighted | bristol_gorillas | 123 | complete | 1.0000 | 1.0000 | descriptive_test_ROC_not_a_deployment_threshold | 84 | 84 | conditional_known_K_balanced_sample | False |
| ApexID_Convnext_Reweighted | bristol_gorillas | 3407 | complete | 1.0000 | 1.0000 | descriptive_test_ROC_not_a_deployment_threshold | 84 | 84 | conditional_known_K_balanced_sample | False |
| ApexID_Convnext_Reweighted | bristol_gorillas | 2026 | complete | 1.0000 | 1.0000 | descriptive_test_ROC_not_a_deployment_threshold | 84 | 84 | conditional_known_K_balanced_sample | False |
| ApexID_Convnext_Reweighted | bristol_gorillas | 1337 | complete | 1.0000 | 1.0000 | descriptive_test_ROC_not_a_deployment_threshold | 84 | 84 | conditional_known_K_balanced_sample | False |
| ApexID_Convnext_Reweighted | elephants | 42 | missing; same-species metrics unavailable | 0.0000 | 1.0000 | no_eligible_positive_and_negative_pairs | 1024 | 0 | conditional_known_K_balanced_sample | False |
| ApexID_Convnext_Reweighted | elephants | 123 | missing; same-species metrics unavailable | 0.0000 | 1.0000 | no_eligible_positive_and_negative_pairs | 1024 | 0 | conditional_known_K_balanced_sample | False |
| ApexID_Convnext_Reweighted | elephants | 3407 | missing; same-species metrics unavailable | 0.0000 | 1.0000 | no_eligible_positive_and_negative_pairs | 1024 | 0 | conditional_known_K_balanced_sample | False |
| ApexID_Convnext_Reweighted | elephants | 2026 | missing; same-species metrics unavailable | 0.0000 | 1.0000 | no_eligible_positive_and_negative_pairs | 1024 | 0 | conditional_known_K_balanced_sample | False |
| ApexID_Convnext_Reweighted | elephants | 1337 | missing; same-species metrics unavailable | 0.0000 | 1.0000 | no_eligible_positive_and_negative_pairs | 1024 | 0 | conditional_known_K_balanced_sample | False |
| ApexID_Convnext_Reweighted | gcn_id | 42 | missing; same-species metrics unavailable | 0.0000 | 1.0000 | no_eligible_positive_and_negative_pairs | 1024 | 0 | conditional_known_K_balanced_sample | False |
| ApexID_Convnext_Reweighted | gcn_id | 123 | missing; same-species metrics unavailable | 0.0000 | 1.0000 | no_eligible_positive_and_negative_pairs | 1024 | 0 | conditional_known_K_balanced_sample | False |
| ApexID_Convnext_Reweighted | gcn_id | 3407 | missing; same-species metrics unavailable | 0.0000 | 1.0000 | no_eligible_positive_and_negative_pairs | 1024 | 0 | conditional_known_K_balanced_sample | False |
| ApexID_Convnext_Reweighted | gcn_id | 2026 | missing; same-species metrics unavailable | 0.0000 | 1.0000 | no_eligible_positive_and_negative_pairs | 1024 | 0 | conditional_known_K_balanced_sample | False |
| ApexID_Convnext_Reweighted | gcn_id | 1337 | missing; same-species metrics unavailable | 0.0000 | 1.0000 | no_eligible_positive_and_negative_pairs | 1024 | 0 | conditional_known_K_balanced_sample | False |
| ApexID_Convnext_Reweighted | jaguar_antonio | 42 | complete | 1.0000 | 1.0000 | descriptive_test_ROC_not_a_deployment_threshold | 405 | 405 | conditional_known_K_balanced_sample | False |
| ApexID_Convnext_Reweighted | jaguar_antonio | 123 | complete | 1.0000 | 1.0000 | descriptive_test_ROC_not_a_deployment_threshold | 405 | 405 | conditional_known_K_balanced_sample | False |
| ApexID_Convnext_Reweighted | jaguar_antonio | 3407 | complete | 1.0000 | 1.0000 | descriptive_test_ROC_not_a_deployment_threshold | 405 | 405 | conditional_known_K_balanced_sample | False |
| ApexID_Convnext_Reweighted | jaguar_antonio | 2026 | complete | 1.0000 | 1.0000 | descriptive_test_ROC_not_a_deployment_threshold | 405 | 405 | conditional_known_K_balanced_sample | False |
| ApexID_Convnext_Reweighted | jaguar_antonio | 1337 | complete | 1.0000 | 1.0000 | descriptive_test_ROC_not_a_deployment_threshold | 405 | 405 | conditional_known_K_balanced_sample | False |

## Interpretation
Higher Rank-1/mAP, pair AUROC/TAR, identity gaps, hard-negative margins and identity silhouette are favorable. Lower hard-margin violation rates are favorable. Representation diagnostics do not replace the closed-set retrieval results.

## Statistical limits
Retrieval intervals are conditional on frozen checkpoints. New diagnostic standard deviations describe overlapping evaluation splits and are not independent training-run confidence intervals.

## Files
See `tables/` for the complete per-model/per-dataset results, metric units, coverage, comparisons, histograms and ROC data.


## Export the paper bundle

In [28]:
import zipfile
bundle_path = OUTPUT_ROOT / "apexid_closed_set_paper_outputs.zip"
with zipfile.ZipFile(bundle_path, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
    for filename in ("final_benchmark_report.md", "final_benchmark_report.json", "benchmark_manifest.json", "stage02_manifest.json", "paper_tables.md", "runtime_report.csv"):
        bundle.write(OUTPUT_ROOT / filename, arcname=filename)
    for folder in ("tables", "plots"):
        for path in sorted((OUTPUT_ROOT / folder).rglob("*")):
            if path.is_file():
                bundle.write(path, arcname=str(path.relative_to(OUTPUT_ROOT)))
print("Closed-set evaluation complete.")
print("Full output:", OUTPUT_ROOT)
print("Paper bundle:", bundle_path)


Closed-set evaluation complete.
Full output: /kaggle/working/apexid_cvpr_closed_representation
Paper bundle: /kaggle/working/apexid_cvpr_closed_representation/apexid_closed_set_paper_outputs.zip
